# ScienceDirect 2026 — "Accessing Content" and sub-contact-reason analysis

Implements `specs/ScienceDirect – "Accessing Content" and further LLM analysis.docx` over the 2026
ScienceDirect tickets in `data/sd_collapsed_2026.csv`.

Two stages, exactly as the specification frames them:

1. **Primary classification** — every ticket is either `Accessing content` or `Anything else`.
2. **Sub-contact reason** — *only* the `Accessing content` tickets get one of six codes
   (`PA`, `INST_INFO`, `INST_ISSUE`, `PURCHASED`, `AVAILABILITY`, `OTHER`).

The specification is emphatic on two points that shape the whole design: classify on the
customer's **actual need** rather than on vocabulary, and **do not infer anything the ticket does
not state**. So every prediction carries a verbatim quote from the transcript, and section 11
audits stage-2 predictions against the specification's own *required markers* lists.

## Why not `data/ScienceDirect_Collapsed.xlsx`

That file was the original input for this analysis. It is **not ScienceDirect data**. It is
row-for-row identical to `data/Evolve_Collapsed.xlsx` — 126,504 rows compared, of which 9 differ,
and those 9 differ only by an empty cell written once as `nan` and once as `""`. Same header, same
first row, same reference numbers.

The content confirms it. Of its 17,729 classifiable 2026 tickets, **79.5% mention Evolve and 0.1%
mention ScienceDirect**; 20.3% name Adaptive Quizzing, HESI or Shadow Health. A 30-ticket smoke
test on it returned nursing-education tickets — course access codes, A2 exam disconnections,
*"Elsevier Adaptive Quizzing for Introduction to Maternity and Pediatric Nursing"*. Running an
article-access specification over those would have produced a confident-looking dataset about the
wrong product, with `PA`, `KBART` and institutional-entitlement codes that mostly cannot apply.

This notebook therefore uses `data/sd_collapsed_2026.csv` — 57.1% of its conversations name
ScienceDirect, 0.1% mention Evolve. It is the same extract notebook 07 classified.

## The source file

`data/sd_collapsed_2026.csv` — 23,344 rows x 5 columns: `INCIDENT_REF`, `Incident_ID`, `Channel`,
`Chat_Start_Date`, `Conversation`.

| | |
|---|---|
| dated 2026 | **23,055** (2026-01-01 to 2026-07-31, 212 days) — the run population |
| dated 2023–2025 | 289 — excluded, written out rather than dropped |
| `INCIDENT_REF` | unique across all 23,344 rows |
| conversation chars, 2026 | mean 2,122 · median 1,349 · p90 4,076 · p99 16,143 · max 247,499 |
| channels | CSS Email 9,379 · Web 9,315 · Chat 1,749 · Phone 1,563 · Email 942 · Fax 106 · Facebook 1 |
| content-free (payload < 40 chars) | 1,014 — assigned by rule, no LLM call |
| sent to the model | **22,041** |

The distribution is long-tailed: the median conversation is 1,349 characters and the longest is
247,499 (~62K tokens). Section 5 caps what is sent at 16,000, which touches 234 rows.

## Relationship to notebook 07

Notebook 07 classified this same extract into a different 13-way contact-reason taxonomy. Nothing
from that run is read here and its output is untouched. This is an independent classification
against the Accessing Content specification, and the two are not comparable ticket by ticket — the
taxonomies answer different questions.

## Order of operations

Sections 0–6 are free and assert their own results. Sections 7 and 8 are the cheap gates that
must pass before the expensive run. **Read the output of section 7 before spending anything more** —
it is a human reading of 30 classifications, and it is the real decision point.

| section | what | cost |
|---|---|---|
| 0–3 | setup, spec transcription, definition workbook, build | $0 |
| 4 | **Gate 1** — read the exact prompts | $0 |
| 5–6 | preprocess tickets, profiles, checkpoint guard | $0 |
| 7 | **Gate 3** — 30-ticket smoke, both stages, read by hand | ~$0.25 |
| 8 | **Gate 4** — 1,000-record scale probe | ~$6 |
| 9 | full stage 1 — 22,041 tickets | ~$125, ~4.7 h |
| 10 | full stage 2 — the Accessing content subset (~6,700) | ~$40, ~1.4 h |
| 11–13 | marker audit, merge, deliverable | $0 |

The last two costs come from a 1,000-ticket probe measured at **$0.00556 per ticket and 1.31
records/second** — figures that depend on prompt length, so they are good to within a few per cent.
The stage-2 volume does not carry over: an earlier probe put `Accessing content` at 30.2% of
tickets, but that was under a longer prompt than the one now in section 1, so section 8 must be
re-run to size stage 2. Everything through section 8 costs about $6.

## Two things that will silently cost you money if ignored

1. **Checkpoint resume ignores prompt changes.** `rca run` resumes from
   `runs/checkpoints/<stem>_checkpoint.jsonl` and reports `skip=N todo=0` with zero LLM calls
   whenever every module already has a result. After an interruption that is exactly right; after
   editing the taxonomy it hands back stale answers. Section 6 fingerprints each module's prompt
   separately and drops only the checkpoints whose prompt actually changed.
2. **`rca build build` rewrites the shared config from its input workbook only.** The live
   `generated/pipeline_config.json` holds one product's modules at a time; after section 3 it holds
   these two. Section 0c backs up whatever was there and writes the restore command.

## What this notebook does not tell you

**There is no accuracy figure here, for either stage.** No human-labelled set exists for this
taxonomy — the specification is new, and nothing in this repository carries `Accessing content` /
`Anything else` labels, let alone the six sub-contact reasons. Nothing below reports precision,
recall or F1, because there is nothing to measure them against.

What the notebook offers instead, and what each is worth:

| check | what it tells you | what it cannot |
|---|---|---|
| §4 prompt review | the model receives the specification verbatim | whether it applies it well |
| §7 30-ticket smoke | the plumbing works, and how the reasoning reads | anything statistical at n=30 |
| §8 1,000-ticket probe | blank-code rate, throughput, class distribution | whether the classes are right |
| evidence grounding | what share of quotes are verbatim in the transcript | whether the quote supports the label |
| §11 marker audit | whether a sub-reason's spec-required markers are present | whether the choice was correct |

Every number this notebook produces is a **count or a distribution**. None is a measure of
correctness. The one thing that would change that is a few hundred hand-labelled tickets; the
deliverable is built to be sampled from for exactly that, and the closing section says how.

This is restated on the deliverable's `Eval methodology` sheet, so nobody reading only the workbook
mistakes a distribution for an accuracy.


## 0. Configuration — edit this cell

In [ ]:
import hashlib
import html
import json
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
import zipfile
from pathlib import Path

import pandas as pd

# -- Project root discovery (walk up until rca_pipeline/ is found) ----------
_root = Path().absolute()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
BASE = _root

# The system Python has pandas built against NumPy 1.x under NumPy 2.x and
# raises "_ARRAY_API not found" on import. Fail now rather than mid-run.
assert pd.__version__.startswith('3.'), (
    f'pandas {pd.__version__} - expected 3.x from .venv. Wrong kernel selected?')

# -- Inputs ----------------------------------------------------------------
# Globbed, not spelled out: the filename carries an en dash and curly quotes.
_specs = sorted((BASE / 'specs').glob('*Accessing Content*.docx'))
assert len(_specs) == 1, f'expected exactly one spec docx, found {[p.name for p in _specs]}'
SPEC_DOCX = _specs[0]

# NOT data/ScienceDirect_Collapsed.xlsx - that file is a mis-named copy of
# data/Evolve_Collapsed.xlsx (identical row for row bar 9 nan-vs-empty cells;
# 79.5% of its conversations mention Evolve, 0.1% mention ScienceDirect).
TICKETS_CSV = BASE / 'data' / 'sd_collapsed_2026.csv'

# -- Build artefacts -------------------------------------------------------
DEFN_XLSX     = BASE / 'definitions' / 'sd_2026_accessing_content_definition.xlsx'
TAXONOMY_DIR  = BASE / 'generated' / 'taxonomies'
MODULES_DIR   = BASE / 'generated' / 'modules'
CONFIG_BACKUP = BASE / 'generated' / '_backup_pre_sd_ac'

# -- Columns ---------------------------------------------------------------
ID_COL   = 'INCIDENT_REF'
TEXT_COL = 'Conversation'
DATE_COL = 'Chat_Start_Date'

# -- Stage 1: primary classification --------------------------------------
MOD_P    = 'sd_ac_primary'
PREFIX_P = 'SDAP'
STUB_P   = 'sdap'

# -- Stage 2: sub-contact reason (Accessing content tickets only) ---------
MOD_S    = 'sd_ac_subreason'
PREFIX_S = 'SDAS'
STUB_S   = 'sdas'

ALL_MODULES     = [MOD_P, MOD_S]
MODULE_PREFIXES = {MOD_P: PREFIX_P, MOD_S: PREFIX_S}

# -- Preprocessing --------------------------------------------------------
TARGET_YEAR          = 2026
SHORT_CHAR_THRESHOLD = 40      # payload chars below which a ticket carries no information
TRUNC_CAP            = 16_000  # per-conversation ceiling sent to the model
TRUNC_HEAD           = 4_000
TRUNC_TAIL           = 11_900  # 11,900 not 12,000: the truncation marker counts toward the cap

# -- LLM tuning -----------------------------------------------------------
# Defined here rather than in section 6 because section 4 folds it into the
# checkpoint fingerprints, and a constant read in section 4 must not be defined
# in section 6. Inherited from the ScienceDirect and LeapSpace runs, which
# completed ~24,000 tickets between them without a retry storm.
TUNING = {
    'batch_size': 5, 'max_chars': 40_000, 'concurrency': 4,
    'retries': 5, 'max_tokens': 8192, 'track_costs': True,
    'cost_report_dir': 'runs/cost_reports',
    # Pinned rather than inherited. RunConfig already defaults to 0.0 via
    # RCA_TEMPERATURE, but a default is not a record: written here, the value that
    # produced the dataset sits in the profile next to its output.
    'temperature': 0.0,
}
# Which of the above change what the model returns, and so must invalidate a
# checkpoint when edited. batch_size and max_chars are in the list because they
# decide which records share a request, and a record batched with different
# neighbours is read in a different context. Throughput settings are excluded, so
# re-tuning concurrency or retries costs nothing.
OUTPUT_AFFECTING = ('temperature', 'max_tokens', 'batch_size', 'max_chars')

# -- Outputs --------------------------------------------------------------
INTER_DIR    = BASE / 'runs' / 'intermediate'
SMOKE_DIR    = BASE / 'runs' / 'smoke_sd_ac_2026'
OUT_DIR      = BASE / 'runs' / 'output'
FEEDBACK_DIR = BASE / 'feedback' / 'sd_ac_2026'
for _d in (INTER_DIR, SMOKE_DIR, OUT_DIR, DEFN_XLSX.parent, BASE / 'profiles'):
    _d.mkdir(parents=True, exist_ok=True)

STAGE1_INPUT  = BASE / 'data' / 'sd_ac_stage1_input_2026.csv'
PROFILE_P     = BASE / 'profiles' / 'sd_2026_ac_primary.json'
PROFILE_S     = BASE / 'profiles' / 'sd_2026_ac_subreason.json'
FINAL_XLSX    = OUT_DIR / 'sd_2026_accessing_content.xlsx'
FINAL_CSV     = OUT_DIR / 'sd_2026_accessing_content_full_text.csv'

# rca run prints U+2192 in its help text and log lines. Under the Windows
# console default (cp1252) that raises UnicodeEncodeError and kills the subprocess,
# so every child process gets UTF-8 forced on.
SUBPROCESS_ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8', 'PYTHONUTF8': '1'}

# A kernel left over from an older version of this notebook can hold some of these
# names and not others, which produces a NameError several cells later that reads
# like a logic bug. Fail here instead, where the cause is obvious.
assert set(OUTPUT_AFFECTING) <= set(TUNING), (
    f'OUTPUT_AFFECTING names keys absent from TUNING: {set(OUTPUT_AFFECTING) - set(TUNING)}')

print(f'Project root : {BASE}')
print(f'Spec         : {SPEC_DOCX.name}')
print(f'Tickets      : {TICKETS_CSV.relative_to(BASE)}')
print(f'Definition   : {DEFN_XLSX.relative_to(BASE)}')
print(f'Modules      : {ALL_MODULES}')

## 0b. Shared helpers

`norm_label` is the one function everything else depends on. The specification `.docx` carries
invisible characters that would otherwise split a category in two: zero-width spaces (U+200B),
non-breaking spaces (U+00A0), and stray leading or trailing spaces. Taxonomy labels are join keys,
so an unnormalised one silently becomes a different category.

Order matters: zero-width characters are deleted **before** NFKC, because NFKC does not remove
them and a stray U+200B inside a word would survive into the key.

In [ ]:
_ZERO_WIDTH = dict.fromkeys(map(ord, '​‌‍﻿'))


def norm_label(value) -> str:
    """Normalise a taxonomy label to a stable join key, preserving its wording."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    text = str(value)
    if text.strip().lower() in ('', 'nan', 'none'):
        return ''
    text = text.translate(_ZERO_WIDTH)          # delete before NFKC
    text = unicodedata.normalize('NFKC', text)  # folds U+00A0 -> ' '
    return re.sub(r'\s+', ' ', text).strip()


def leaf_labels(module: str) -> dict:
    """code -> label for every leaf of a generated taxonomy JSON."""
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    return {code: leaf['label']
            for node in tax.values()
            for code, leaf in node.get('children', {}).items()}


def read_stage_output(path) -> pd.DataFrame:
    """Read a stage output, keeping the id column as a clean string."""
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower().startswith('.xls') else pd.read_csv(path)
    df[ID_COL] = df[ID_COL].astype(str).str.strip()
    return df


def run_cmd(cmd, label):
    """Run a short command, echoing its output; raise on failure."""
    print(f'\n$ {" ".join(str(c) for c in cmd)}')
    r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
    print(r.stdout or '', end='')
    print(r.stderr or '', end='')
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed (exit {r.returncode})')


print('Helpers defined.')

### Self-test for `norm_label`

Each case below is a real shape seen in Elsevier taxonomy exports. If one of these regresses, a
join key silently changes and a whole category goes missing without an error.

In [ ]:
_CASES = [
    ('Set up teams or departments​', 'Set up teams or departments'),  # zero-width space
    (' Content issue', 'Content issue'),                                   # leading space
    ('Admin/User\xa0management issue', 'Admin/User management issue'),      # non-breaking space
    ('Browser compatibility issues ​', 'Browser compatibility issues'),
    ('Accessing  content', 'Accessing content'),                           # doubled space
    (None, ''), ('nan', ''), ('   ', ''),
]
for raw, want in _CASES:
    got = norm_label(raw)
    assert got == want, f'norm_label({raw!r}) -> {got!r}, expected {want!r}'
print(f'norm_label: all {len(_CASES)} cases pass.')

## 0c. Safety net — back up the shared config

`rca build build` rebuilds `generated/pipeline_config.json` from its input workbook only,
so any module not in that workbook is dropped. Whatever product's modules are live right now — the
four ClinicalKey / CSD ones from notebook 09, at the time of writing — will be gone after
section 3.

That is acceptable, because those runs are finished and their outputs are saved under
`runs/output/`, but the backup makes it reversible. It writes to a **new** directory and never
clobbers a file already there: overwriting an earlier backup with the state it was meant to
protect you from is the one way a safety net makes things worse.

In [ ]:
CONFIG_BACKUP.mkdir(parents=True, exist_ok=True)
_live = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
_live_modules = [m for m in _live['modules'] if m not in ALL_MODULES]

_backed_up = []
for _src in [BASE / 'generated' / 'pipeline_config.json',
             BASE / 'generated' / 'build_manifest.json',
             *[TAXONOMY_DIR / f'{m}.json' for m in _live_modules]]:
    if _src.exists():
        _dst = CONFIG_BACKUP / _src.name
        if not _dst.exists():                 # never clobber an earlier backup
            shutil.copy2(_src, _dst)
        _backed_up.append(_src.name)

print(f'Backup dir: {CONFIG_BACKUP.relative_to(BASE)}')
for _n in _backed_up:
    print(f'  {_n}')
print(f'\nModules in the live config right now: {list(_live["modules"])}')

_restore = ('rca build --input '
            'definitions/csd_2026_pipeline_definition.xlsx --force-all')
(CONFIG_BACKUP / 'RESTORE.txt').write_text(
    'The modules that were live in generated/pipeline_config.json before this notebook ran:\n'
    f'    {", ".join(_live_modules) or "(none)"}\n\n'
    'To put the ClinicalKey/CSD ones back:\n\n'
    f'    {_restore}\n\n'
    'Their outputs are already saved under runs/output/csd_2026_*.\n'
    'The config holds one product at a time; restoring CSD drops the sd_ac_* modules.\n',
    encoding='utf-8')
print(f'\nTo restore the CSD modules later:\n    {_restore}')

## 1. Transcribe the specification, then prove the transcription

The taxonomy lives in a Word document rather than a workbook, so its wording is transcribed into
Python constants below. Every one of them is the **verbatim** text of a paragraph in
`specs/ScienceDirect – "Accessing Content" and further LLM analysis.docx`.

A transcription that silently drifts from its source is the failure mode that costs the most and
shows up the latest, so this cell does not take the transcription on trust. It re-extracts
`word/document.xml` from the live `.docx` and asserts that **every transcribed string is an exact
paragraph of the document**. Edit the specification and this cell fails, rather than classifying
23,055 tickets against a stale taxonomy.

Two composites are assembled from those verbatim parts:

- Each primary label's `Definition` is its intro line followed by its bullets, joined with a
  space. Nothing is reworded, reordered or dropped — the bullets already carry their own
  semicolons, so they read as one sentence.
- `PRINCIPLES` is the document's four closing *Classification principles*, in order.

**One thing in the specification is incomplete.** The `OTHER` row of the sub-contact-reason table
ends mid-sentence: *"When the ticket lacks the required markers for a sub-contact reason,"*. The
Classification principles block below it finishes the same sentence, so the finished version is
used and the truncated cell is recorded in `SPEC_DEFECTS` for the taxonomy owner.

### Two rules are authored, not transcribed

Both are marked in the `notes` column of the definition workbook and printed below, so a reader
can always tell the specification's words from ours.

1. **A decision order for the primary classification.** The specification lists what belongs to
   `Anything else` as a flat set of bullets and warns that the word "access" alone is not enough,
   but it never says which test to apply first. A ticket that says *"I cannot log in to download
   this article"* satisfies a bullet on both sides. The authored rule makes the obstacle decide,
   and in the order the specification's own exclusions imply. It carries two clauses the 30-ticket
   smoke test in section 7 showed were needed: the exclusions apply only to a cause the ticket
   **states** — *"I cannot view the whole article"* names no browser and no network, so it stays
   `Accessing content` — and a Patient Access request is `Accessing content`, not an
   administrative one, since the specification makes `PA` a sub-contact reason *of* `Accessing
   content`.
2. **An evidence threshold for the sub-contact reason.** The specification requires explicit
   markers per code and says to fall back to `OTHER` when they are absent, but a model reading six
   definitions will still stretch a weak signal into a specific code. The authored rule states the
   threshold as an instruction and, importantly, tells the model that `OTHER` is a correct answer
   rather than a failure code.

In [ ]:
def extract_spec_paragraphs(path) -> list:
    """Paragraph text of a .docx, normalised. No new dependency: the file is a zip."""
    xml = zipfile.ZipFile(path).read('word/document.xml').decode('utf-8')
    xml = xml.replace('</w:p>', '</w:p>\n')       # one paragraph per line
    xml = re.sub(r'<w:tab[^>]*/>', '\t', xml)
    xml = re.sub(r'<w:br[^>]*/>', '\n', xml)      # manual line breaks
    text = html.unescape(re.sub(r'<[^>]+>', '', xml))
    return [p for p in (norm_label(l) for l in text.split(chr(10))) if p]


SPEC_PARAS = extract_spec_paragraphs(SPEC_DOCX)
print(f'{SPEC_DOCX.name}: {len(SPEC_PARAS)} non-empty paragraphs')

# ---- Global instructions (apply to both stages) -------------------------
G_THREAD = 'Analyse each ScienceDirect ticket using the full conversation thread.'
G_ONLYIF = 'Only if the primary classification is Accessing content, assign one of the five sub-contact reasons below. Do not assign a sub-contact reason to tickets classified as Anything else.'
G_NEED   = 'Classify based on the customer’s actual need. Agent messages can confirm context, but do not infer information that is not explicit in the ticket.'

# ---- 1. Primary classification -----------------------------------------
AC_INTRO = 'Use this label only when the customer is trying to:'
AC_BULLETS = [
    'obtain, read, download, or receive a specific article, journal, issue, archive item, content package, or supporting material;',
    'confirm whether they or their institution can access specific content;',
    'resolve an issue accessing content they expect to be entitled to; or',
    'access content they have personally purchased.',
]
AE_INTRO = 'Use this label for all other contacts, including:'
AE_BULLETS = [
    'password, login, ID+, account activation, account switching, or SSO issues;',
    'cache, cookies, browser, device, VPN, proxy, or network issues;',
    'IP restrictions, Content Protection blocks, security controls, or high-download blocks;',
    'profile, billing, refund, cancellation, or general account-administration requests;',
    'ScienceDirect AI requests;',
    'requests that do not concern access to specific content.',
]
AE_WORD = 'The word “access” alone is not enough to classify a ticket as Accessing content.'

PRIMARY_DEFS = {
    'Accessing content': ' '.join([AC_INTRO, *AC_BULLETS]),
    'Anything else': ' '.join([AE_INTRO, *AE_BULLETS, AE_WORD]),
}
PRIMARY_ORDER = list(PRIMARY_DEFS)          # fixes the codes: SDAP1.1, SDAP2.1
ACCESSING_CONTENT = 'Accessing content'
ANYTHING_ELSE     = 'Anything else'

# ---- 2. Sub-contact reasons (spec code, name, definition and markers) ---
SUB_DEFS = [
    ('PA',
     'Request an article through Patient Access',
     'The customer explicitly mentions Patient Access, identifies as a patient or non-professional caregiver, or the agent explicitly confirms that the request is handled through Patient Access. Do not use for healthcare professionals, researchers, pharmacists, or veterinarians unless Patient Access is explicit.'),
    ('INST_INFO',
     'Check whether an institution subscribes to a specific title, year or package',
     'A named institution is asking whether it has access to specific content. Markers include: “our institution”, “our library”, subscription, licence, entitlement, holdings, coverage, KBART, title inclusion, package, or coverage years. Use this only where the customer is seeking clarification, rather than reporting a confirmed entitlement failure.'),
    ('INST_ISSUE',
     'Access content that the institution is entitled to, but which is not available to the user',
     'A named institution is involved and the ticket shows that access is expected but not delivered. Markers include: “we are subscribed”, “we have paid”, “we should have access”, invoice, agreement, entitlement, ECR, account mapping, institutional token, provisioned title, or agent confirmation of entitlement.'),
    ('PURCHASED',
     'Access an article or subscription purchased directly from ScienceDirect',
     'The customer refers to a personal purchase, order number, receipt, payment, pay-per-view purchase, personal subscription, or order history, and cannot access the purchased content. Do not assume a purchase without an explicit commercial marker.'),
    ('AVAILABILITY',
     'Request a missing article, archive item or supplementary material',
     'The requested content cannot be supplied because it is unavailable, missing, not digitised, or lacks a supplementary/companion file, appendix, dataset, or other supporting material.'),
    ('OTHER',
     'Other',
     'When the ticket lacks the required markers for a sub-contact reason, classify it as « other »'),
]
SUB_LABELS = [f'{c} - {n}' for c, n, _d in SUB_DEFS]   # e.g. 'PA - Request an article...'
SUB_CODE_OF_LABEL = {lab: c for lab, (c, _n, _d) in zip(SUB_LABELS, SUB_DEFS)}

# ---- Classification principles ------------------------------------------
PRINCIPLE_PARTS = [
    'Assign only one sub-contact reason to each Accessing content ticket.',
    'Use the most specific category supported by explicit evidence.',
    'Do not infer whether a customer is institutional or individual from their email address or organisation name alone.',
    'When the ticket lacks the required markers for a sub-contact reason, classify it as « other »',
]


def end_sentence(p: str) -> str:
    """Add a full stop where a spec paragraph lacks one.

    The document is bullets and table cells, several of which end with no
    terminator - the last Classification principle ends on the closing guillemet
    of << other >>. The prompt concatenates them into prose, so without this the
    last principle runs straight into the next sentence. Punctuation only; not
    one word is changed, and the untouched originals are what the fidelity check
    above compares against.
    """
    return p if p.endswith(('.', '?', '!', ':', ';')) else p + '.'


PRINCIPLES = ' '.join(end_sentence(p) for p in PRINCIPLE_PARTS)

# ---- Verify every transcribed string against the live document ----------
_TRANSCRIBED = [G_THREAD, G_ONLYIF, G_NEED, AC_INTRO, *AC_BULLETS, AE_INTRO,
                *AE_BULLETS, AE_WORD, *PRINCIPLE_PARTS,
                *[n for _c, n, _d in SUB_DEFS if n != 'Other'],
                *[d for _c, _n, d in SUB_DEFS]]
_missing = [s for s in _TRANSCRIBED if s not in SPEC_PARAS]
assert not _missing, ('these strings are no longer paragraphs of the spec document - the spec changed or the transcription drifted:\n  '
                     + '\n  '.join(repr(m[:110]) for m in _missing))
print(f'Spec fidelity: all {len(_TRANSCRIBED)} transcribed strings are exact paragraphs of the document.')

# ---- Defects found in the source document -------------------------------
SPEC_DEFECTS = [
    {
        'where': 'Sub-contact reasons table, OTHER row',
        'text': 'When the ticket lacks the required markers for a sub-contact reason,',
        'issue': 'Sentence is cut off mid-clause; it has no main verb.',
        'resolution': ('Used the completed form of the same sentence from the Classification '
                       'principles block: ' + repr(PRINCIPLE_PARTS[3])),
    },
    {
        'where': 'Section 2 heading and the line above it',
        'issue': ('The document says five sub-contact reasons but the table lists six rows, '
                  'the sixth being OTHER. Six codes are built; OTHER is the residual class, '
                  'which is consistent with the closing principle.'),
        'resolution': 'Built six leaf codes.',
    },
    {
        'where': 'Section 1 Anything else bullet 3, versus the INST_ISSUE marker list',
        'issue': ('The Anything else bullet excludes IP restrictions and security controls, '
                  'while INST_ISSUE names ECR, agreement, entitlement and provisioned title '
                  'among its required markers. An institutional IP-range or title '
                  'provisioning request carries both, so the two readings send the same '
                  'ticket to opposite primary classes - and these are high-volume tickets.'),
        'resolution': ('UNRESOLVED. The prompt takes no side: it carries the Anything else '
                       'bullet and the INST_ISSUE marker list exactly as the specification '
                       'writes them, and the model decides. Expect these tickets to split '
                       'between the two primary classes until the taxonomy owner rules on it. '
                       'THIS NEEDS A DECISION.'),
    },
]
for _d in SPEC_DEFECTS:
    print(f"\nSPEC DEFECT - {_d['where']}\n  {_d['issue']}\n  -> {_d['resolution']}")

### The two authored rules

Printed in full so they can be read next to the specification's own words above.

Both are deliberately thin. Earlier versions of the decision order carried extra clauses added
after reading gate output — a Patient Access carve-out, three named shapes of security block, and a
rule for institutional IP provisioning. Those have been **removed**: they were reactions to
individual tickets rather than readings of the specification, and the classification should stand or
fall on the document as written. What remains is the ordering the specification's own exclusion
bullets imply, and nothing else.

In [ ]:
# AUTHORED - not in the specification. Flagged in the notes column of the
# definition workbook and in the deliverable.
PRIMARY_PRECEDENCE = (
    'Decide on what the customer actually needs, not on the vocabulary they use. The word '
    'access on its own is never enough. Apply these tests in order. First, if what blocks '
    'the customer is a credential or a sign-in - a password, a login, ID+, account '
    'activation, account switching, or SSO - the answer is Anything else, even when a '
    'specific article is named. Second, if what blocks them is the machine or the network - '
    'cache, cookies, a browser, a device, a VPN or a proxy - or an explicit block such as '
    'an IP restriction, a Content Protection block, a security control or a high-download '
    'block, the answer is Anything else. Third, if the request is administrative or '
    'commercial - a profile change, billing, an invoice, a refund, a cancellation - or '
    'concerns ScienceDirect AI, the answer is Anything else. Only when none of those apply, '
    'ask whether the customer is trying to obtain, read, download or receive specific '
    'content, to confirm whether they or their institution can reach specific content, to '
    'resolve a failure to reach content they expect to be entitled to, or to reach content '
    'they bought themselves. If so, the answer is Accessing content. If the ticket does not '
    'say enough to place it either way, answer Anything else rather than assuming a content '
    'request.'
)

SUB_EVIDENCE_RULE = (
    'Each of the five specific sub-contact reasons lists the markers that must be present. '
    'Choose one of them only when at least one of its markers is explicit in the ticket, '
    'and quote that marker as the evidence. Do not infer an institution from an email '
    'domain or an organisation name, do not infer a purchase without an explicit commercial '
    'marker, and do not infer Patient Access from the fact that the writer may be unwell or '
    'is asking about a medical topic. When the required markers are absent, answer OTHER. '
    'OTHER is the correct answer for a genuine Accessing content ticket that simply does '
    'not carry the markers - it is not a failure code, and it is what the specification '
    'asks for.'
)

for _name, _rule in (('PRIMARY_PRECEDENCE', PRIMARY_PRECEDENCE),
                     ('SUB_EVIDENCE_RULE', SUB_EVIDENCE_RULE)):
    assert _rule not in SPEC_PARAS, f'{_name} is spec text, not authored - move it'
    print(f'{_name} ({len(_rule)} chars)\n{_rule}\n')

## 2. Write the pipeline definition workbook

`rca build` consumes three sheets — `modules`, `fields`, `taxonomy` — and **assigns
taxonomy codes by row order**, so codes are never written by hand. `PRIMARY_ORDER` therefore fixes
`SDAP1.1` / `SDAP2.1`, and `SUB_DEFS` order fixes `SDAS1.1` … `SDAS6.1`.

Both taxonomies are two-level because `rca build` unconditionally appends *"Always use
LEAF-level codes"* to the generated prompt, so an L1-only code would contradict its own
instructions. Each category is an L1 group with a single L2 leaf repeating the label, and the
definition sits on the **leaf only** — `taxonomy_to_rich_prompt` prints both levels, so text on
both would duplicate every definition in the prompt.

**Sub-contact-reason labels carry the specification's code**, e.g. `PA - Request an article through
Patient Access`. The alternative was an explicit `code` column (`rca build` supports one)
set to `PA`, `INST_INFO`, and so on. That was rejected: `leaf_label_lookup` merges L1 and leaf
codes into one namespace, so a group and its only child cannot share a code, and the six spec
codes would need six invented parent codes with no meaning. Putting the code in the label keeps it
visible in the prompt, in the model's answer, and in the deliverable, with one mapping
(`SUB_CODE_OF_LABEL`) instead of a parallel code space.

Three fields per module, mirroring the `sdcr_*` fields of the existing ScienceDirect definition:
the code, a justification naming the nearest rejected alternative, and a **verbatim** quote. The
quote is not decoration — it is what makes the specification's "do not infer" rule auditable, and
section 12 reads it.

In [ ]:
# ---- modules sheet -------------------------------------------------------
MOD_DESC_P = (
    'Classify each ScienceDirect support ticket into exactly one of two primary classifications: '
    f'{ACCESSING_CONTENT} or {ANYTHING_ELSE}. '
    f'{G_THREAD} {G_NEED} '
    f'Assign the single best-fitting LEAF code (e.g. {PREFIX_P}1.1). '
    f'{PRIMARY_PRECEDENCE}')

MOD_DESC_S = (
    f'This ticket has already been classified as {ACCESSING_CONTENT}. Assign exactly one '
    'sub-contact reason from the taxonomy below, each of which carries its definition and the '
    'markers that must be present for it to apply. '
    f'{G_THREAD} {G_NEED} {PRINCIPLES} '
    f'Assign the single best-fitting LEAF code (e.g. {PREFIX_S}1.1). '
    f'{SUB_EVIDENCE_RULE}')

_AUTHORED_NOTE = 'Contains an AUTHORED rule, not present in the source specification.'

modules_rows = [
    {'module_name': MOD_P, 'taxonomy_prefix': PREFIX_P,
     'taxonomy_file': f'{MOD_P}.json', 'reorder_group': 'A', 'reorder_order': 1,
     'enabled': 'TRUE', 'description': MOD_DESC_P,
     'notes': f'Stage 1 of 2. {_AUTHORED_NOTE} See PRIMARY_PRECEDENCE.'},
    {'module_name': MOD_S, 'taxonomy_prefix': PREFIX_S,
     'taxonomy_file': f'{MOD_S}.json', 'reorder_group': 'B', 'reorder_order': 2,
     'enabled': 'TRUE', 'description': MOD_DESC_S,
     'notes': (f'Stage 2 of 2 - runs only on tickets stage 1 called {ACCESSING_CONTENT}. '
               f'{_AUTHORED_NOTE} See SUB_EVIDENCE_RULE.')},
]

# ---- fields sheet --------------------------------------------------------
def _fields_for(stub, prefix, what, alternative_phrase):
    return [
        {'module_name': None, 'field_name': f'{stub}_code', 'field_type': 'code',
         'description': 'Assigned leaf code',
         'prompt_instruction': (
             f'Assign the single leaf-level code (e.g. {prefix}1.1) for the {what} that best '
             'matches this ticket. Read each Definition in full and match on what the ticket '
             'shows, not on surface keywords. Only use codes from the taxonomy.'),
         'in_output': 'TRUE', 'explode_list': 'FALSE', 'validate_enum': 'TRUE', 'notes': None},
        {'module_name': None, 'field_name': f'{stub}_justification', 'field_type': 'str',
         'description': 'Why this code was chosen over the nearest alternative',
         'prompt_instruction': (
             f'In one or two sentences, explain why this {what} applies and why '
             f'{alternative_phrase} does not. Refer to what the ticket actually shows. Do not use '
             'apostrophes or quotation marks in your answer.'),
         'in_output': 'TRUE', 'explode_list': 'FALSE', 'validate_enum': 'FALSE', 'notes': None},
        {'module_name': None, 'field_name': f'{stub}_evidence', 'field_type': 'str',
         'description': 'Verbatim supporting quote from the transcript',
         'prompt_instruction': (
             'Copy verbatim the single sentence or phrase from the transcript that most strongly '
             'supports this choice. Do not paraphrase, infer, or add words that are not in the '
             'transcript. If no direct evidence exists, return an empty string.'),
         'in_output': 'TRUE', 'explode_list': 'FALSE', 'validate_enum': 'FALSE', 'notes': None},
    ]


fields_rows = []
for _mod, _stub, _prefix, _what, _alt in (
        (MOD_P, STUB_P, PREFIX_P, 'primary classification', 'the other classification'),
        (MOD_S, STUB_S, PREFIX_S, 'sub-contact reason', 'the closest alternative')):
    for _f in _fields_for(_stub, _prefix, _what, _alt):
        _f['module_name'] = _mod
        fields_rows.append(_f)

# ---- taxonomy sheet ------------------------------------------------------
# L1 group carries the label only; the definition goes on the leaf, because
# taxonomy_to_rich_prompt prints both levels.
taxonomy_rows = []
for _label in PRIMARY_ORDER:
    taxonomy_rows.append({'module_name': MOD_P, 'level': 'L1', 'label': _label,
                          'description': None, 'notes': None})
    taxonomy_rows.append({'module_name': MOD_P, 'level': 'L2', 'label': _label,
                          'description': f'Definition: {PRIMARY_DEFS[_label]}',
                          'notes': 'Verbatim from the specification, section 1.'})

for (_spec_code, _name, _defn), _label in zip(SUB_DEFS, SUB_LABELS):
    _note = 'Verbatim from the specification, section 2 table.'
    if _spec_code == 'OTHER':
        _note = ('Definition taken from the Classification principles block: the OTHER row of '
                 'the specification table is cut off mid-sentence. See SPEC_DEFECTS.')
    taxonomy_rows.append({'module_name': MOD_S, 'level': 'L1', 'label': _label,
                          'description': None, 'notes': None})
    taxonomy_rows.append({'module_name': MOD_S, 'level': 'L2', 'label': _label,
                          'description': f'Definition and required markers: {end_sentence(_defn)}',
                          'notes': _note})

# ---- write ---------------------------------------------------------------
_COLS = {
    'modules': ['module_name', 'taxonomy_prefix', 'taxonomy_file', 'reorder_group',
                'reorder_order', 'enabled', 'description', 'notes'],
    'fields': ['module_name', 'field_name', 'field_type', 'description', 'prompt_instruction',
               'in_output', 'explode_list', 'validate_enum', 'notes'],
    'taxonomy': ['module_name', 'level', 'label', 'description', 'notes'],
}
_frames = {'modules': pd.DataFrame(modules_rows)[_COLS['modules']],
           'fields': pd.DataFrame(fields_rows)[_COLS['fields']],
           'taxonomy': pd.DataFrame(taxonomy_rows)[_COLS['taxonomy']]}

with pd.ExcelWriter(DEFN_XLSX, engine='openpyxl') as _xw:
    for _sheet, _df in _frames.items():
        _df.to_excel(_xw, sheet_name=_sheet, index=False)

# ---- round-trip: what rca build will actually read --------------
_back = pd.read_excel(DEFN_XLSX, sheet_name=None, dtype=str)
assert set(_back) == {'modules', 'fields', 'taxonomy'}, f'unexpected sheets: {sorted(_back)}'
assert len(_back['modules']) == 2, f'expected 2 modules, wrote {len(_back["modules"])}'
assert len(_back['fields']) == 6, f'expected 6 fields, wrote {len(_back["fields"])}'
assert len(_back['taxonomy']) == 2 * (len(PRIMARY_ORDER) + len(SUB_DEFS)), 'taxonomy row count'

# Excel silently truncates a cell over 32,767 chars. The module descriptions are the
# only long cells here; check them rather than discovering a cut prompt in section 4.
for _s, _df in _back.items():
    for _c in _df.columns:
        _long = _df[_c].dropna().astype(str).str.len().max()
        assert pd.isna(_long) or _long < 32_000, f'{_s}.{_c} is {_long} chars - near the Excel cap'

# Every definition must have survived the write unchanged.
_tax_back = _back['taxonomy']
for _label in PRIMARY_ORDER:
    _row = _tax_back[(_tax_back.level == 'L2') & (_tax_back.label == _label)]
    assert len(_row) == 1 and PRIMARY_DEFS[_label] in _row.iloc[0]['description'], \
        f'primary definition did not survive the write: {_label}'
for (_spec_code, _n, _defn), _label in zip(SUB_DEFS, SUB_LABELS):
    _row = _tax_back[(_tax_back.level == 'L2') & (_tax_back.label == _label)]
    assert len(_row) == 1 and _defn in _row.iloc[0]['description'], \
        f'sub definition did not survive the write: {_spec_code}'

print(f'Wrote {DEFN_XLSX.relative_to(BASE)}')
for _s, _df in _frames.items():
    print(f'  {_s:<9} {len(_df):>2} rows')
print(f'\nPrimary labels    : {PRIMARY_ORDER}')
print('Sub-reason labels :')
for _label in SUB_LABELS:
    print(f'  {_label}')

## 3. Build the pipeline artefacts

Dry run first — it reports what would be written without touching anything.

In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--dry-run'], 'build_pipeline --dry-run')

In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--force', *ALL_MODULES], 'build_pipeline')

# ---- Verify the built shape ---------------------------------------------
cfg = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
assert set(cfg['modules']) == set(ALL_MODULES), (
    f'config modules mismatch\n  missing: {set(ALL_MODULES) - set(cfg["modules"])}'
    f'\n  extra  : {set(cfg["modules"]) - set(ALL_MODULES)}')

expected_leaves = {MOD_P: len(PRIMARY_ORDER), MOD_S: len(SUB_DEFS)}
for module, want in expected_leaves.items():
    assert (TAXONOMY_DIR / f'{module}.json').exists(), f'missing taxonomy for {module}'
    assert (MODULES_DIR / f'{module}.py').exists(), f'missing {module}.py'
    got = len(leaf_labels(module))
    assert got == want, f'{module}: expected {want} leaves, got {got}'
    assert cfg['modules'][module]['taxonomy_prefix'] == MODULE_PREFIXES[module]

# Field stubs must not collide with the prefixes drop_enrichment_cols() strips,
# or the results would vanish from the output on a re-run.
from rca_pipeline.utils import _STRUCTURAL_PREFIXES
for stub in (STUB_P, STUB_S):
    for bad in _STRUCTURAL_PREFIXES:
        assert not f'{stub}_'.startswith(bad), f'field stub {stub}_ collides with {bad!r}'

# Prefixes are parsed with a greedy ^[A-Z]+ elsewhere in the pipeline, so
# neither may be a prefix of the other.
assert not PREFIX_P.startswith(PREFIX_S) and not PREFIX_S.startswith(PREFIX_P), \
    f'{PREFIX_P} and {PREFIX_S} overlap'

PRIMARY_LABELS = leaf_labels(MOD_P)                     # code -> label
SUB_TAX_LABELS = leaf_labels(MOD_S)
CODE_OF_PRIMARY = {v: k for k, v in PRIMARY_LABELS.items()}
CODE_OF_SUB = {v: k for k, v in SUB_TAX_LABELS.items()}
assert len(CODE_OF_PRIMARY) == len(PRIMARY_LABELS), 'duplicate primary labels'
assert len(CODE_OF_SUB) == len(SUB_TAX_LABELS), 'duplicate sub-reason labels'
assert set(SUB_TAX_LABELS.values()) == set(SUB_LABELS), 'built sub labels differ from SUB_LABELS'

AC_CODE = CODE_OF_PRIMARY[ACCESSING_CONTENT]            # the code that routes to stage 2
# taxonomy code -> the specification's own code, for the deliverable
SPEC_CODE_OF_TAX_CODE = {code: SUB_CODE_OF_LABEL[label]
                         for code, label in SUB_TAX_LABELS.items()}

print(f'\nBuild verified: {len(ALL_MODULES)} modules, '
      f'{sum(len(leaf_labels(m)) for m in ALL_MODULES)} leaves total.\n')
print('Primary classification codes:')
for _code, _label in PRIMARY_LABELS.items():
    _tag = ' -> routes to stage 2' if _code == AC_CODE else ''
    print(f'  {_code:<9} {_label}{_tag}')
print('\nSub-contact reason codes:')
for _code, _label in SUB_TAX_LABELS.items():
    print(f'  {_code:<9} {SPEC_CODE_OF_TAX_CODE[_code]:<13} {_label}')

## 4. Gate 1 — review the exact prompts *(free, read this output)*

Renders both system prompts exactly as the model will receive them, then asserts that every
definition transcribed in section 1 and both authored rules survive into the prompt **verbatim**.
A dropped definition or a mis-wired prefix is invisible at every later stage and shows up only as
poor accuracy, so this is the cheapest place to catch it.

`importlib.reload` matters: after the rebuild in section 3 the old module object is still in
`sys.modules`, and without the reload the prompts shown here would be the previous build's.

The fingerprints are stored beside each checkpoint in section 6. They are per-module on purpose —
a single global hash would force a full re-run for a one-module fix.

In [ ]:
import importlib

from rca_pipeline.taxonomy import taxonomy_to_rich_prompt

importlib.invalidate_caches()
prompts = {}
for module in ALL_MODULES:
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    name = f'generated.modules.{module}'
    mod = importlib.reload(sys.modules[name]) if name in sys.modules \
        else importlib.import_module(name)
    prompts[module] = mod._build_system_prompt(
        taxonomy_to_rich_prompt(tax, MODULE_PREFIXES[module]))

# ---- Verbatim spec check ------------------------------------------------
checked = 0
for label, defn in PRIMARY_DEFS.items():
    assert label in prompts[MOD_P], f'primary label missing from the prompt: {label!r}'
    assert defn in prompts[MOD_P], f'primary definition missing: {defn[:70]!r}'
    checked += 2
for (spec_code, _name, defn), label in zip(SUB_DEFS, SUB_LABELS):
    assert label in prompts[MOD_S], f'sub label missing from the prompt: {label!r}'
    assert defn in prompts[MOD_S], f'sub definition missing for {spec_code}: {defn[:70]!r}'
    checked += 2
for text, module, what in ((G_THREAD, MOD_P, 'full-thread instruction'),
                           (G_NEED, MOD_P, 'actual-need instruction'),
                           (G_THREAD, MOD_S, 'full-thread instruction'),
                           (G_NEED, MOD_S, 'actual-need instruction'),
                           (PRINCIPLES, MOD_S, 'classification principles'),
                           (PRIMARY_PRECEDENCE, MOD_P, 'authored precedence rule'),
                           (SUB_EVIDENCE_RULE, MOD_S, 'authored evidence rule')):
    assert text in prompts[module], f'{what} missing from {module}'
    checked += 1
print(f'Verbatim check: all {checked} strings present in the prompts as written.\n')

# ---- Per-module fingerprints -------------------------------------------
PROMPT_FINGERPRINTS = {
    m: hashlib.sha256(p.encode('utf-8')).hexdigest()[:16] for m, p in prompts.items()}

# A prompt is not the only thing that changes the answer. Sampling settings do
# too, and so does batch composition - two records in one request see each other's
# context. Fold them in, or editing temperature would silently reuse answers
# sampled at the old one. Both constants come from the section 0 config cell.
_tuning_sig = '|'.join(f'{k}={TUNING[k]}' for k in OUTPUT_AFFECTING)
_preproc = f'{SHORT_CHAR_THRESHOLD}|{TRUNC_CAP}|{TRUNC_HEAD}|{TRUNC_TAIL}|{TARGET_YEAR}'
for _m in ALL_MODULES:
    _extra = _tuning_sig + ('|' + _preproc if _m == MOD_P else '')
    PROMPT_FINGERPRINTS[_m] = hashlib.sha256(
        (prompts[_m] + _extra).encode('utf-8')).hexdigest()[:16]
print(f'\nOutput-affecting tuning folded in: {_tuning_sig}')

print(f'{"module":<18} {"chars":>7} {"~tokens":>8}  fingerprint')
for module in ALL_MODULES:
    p = prompts[module]
    print(f'{module:<18} {len(p):>7,} {len(p) // 4:>8,}  {PROMPT_FINGERPRINTS[module]}')

In [ ]:
print('=' * 78)
print(f'FULL STAGE 1 SYSTEM PROMPT  ({MOD_P})')
print('=' * 78)
print(prompts[MOD_P])

In [ ]:
print('=' * 78)
print(f'FULL STAGE 2 SYSTEM PROMPT  ({MOD_S})')
print('=' * 78)
print(prompts[MOD_S])

## 5. Gate 2 — preprocess the tickets *(free)*

Three transformations, each asserted:

**Filter to calendar 2026.** The file spans 2023-10 to 2026-07 but is overwhelmingly 2026; the 289
earlier rows are written out rather than dropped silently, so anyone extending the run backwards
starts from a file rather than from scratch.

**Short-ticket rule.** Length is measured on the *payload*, with the `Customer:` / `Agent Notes:`
markers stripped, so a ticket padded past the threshold by its markers alone is still caught.
These 1,014 rows get `Anything else` by rule with no LLM call.

That default deserves a defence. The specification has no "not enough info" outcome — only
`Accessing content` and `Anything else` — and it forbids inferring anything the ticket does not
state. A ticket reading `Customer: .` or `Customer: Ghost Call` states nothing, so it cannot
evidence a content request, and `Anything else` is the only faithful answer. `classification_source`
keeps these rows separable for anyone who wants them out of a denominator, and the deliverable
reports their count.

**Truncation to a 16,000-char ceiling, head 4,000 + tail 11,900.** `make_batches` never truncates:
it only *flushes* batches, so a single oversized row is appended and sent whole — and the longest
conversation here is 247,499 characters, roughly 62K tokens. The tail is weighted because the
resolution lives at the end of a transcript, but the customer's original ask is in the first few
hundred characters and stage 1 needs it. The `[...TRUNCATED...]` marker is visible so the model
does not read a splice as continuous, and it counts toward the ceiling — hence 11,900 rather than
12,000, which would land at 16,040 and break the cap it was meant to enforce.

This touches 234 of 23,055 rows while keeping about 93% of all text. Its real purpose is to **bound
the worst-case batch**: with `batch_size 5` and `max_chars 40000`, no request can exceed ~20K input
tokens whatever mix of rows it draws.

In [ ]:
_t0 = time.time()
raw = pd.read_csv(TICKETS_CSV, dtype=str, low_memory=False)
print(f'{TICKETS_CSV.name}: {len(raw):,} rows x {len(raw.columns)} cols '
      f'({time.time() - _t0:.0f}s)')
print(f'  columns: {list(raw.columns)}')

raw[ID_COL] = raw[ID_COL].astype(str).str.strip()
assert raw[ID_COL].is_unique, f'duplicate {ID_COL} in the source file'
# No column may collide with the prefixes drop_enrichment_cols() strips, or it
# would vanish from a pipeline input without an error.
from rca_pipeline.utils import _STRUCTURAL_PREFIXES
_collide = [c for c in raw.columns
            if any(c.lower().startswith(p) for p in _STRUCTURAL_PREFIXES)]
assert not _collide, f'these source columns would be stripped as enrichment: {_collide}'

# Sanity check on provenance: this must be ScienceDirect data, not Evolve.
_sd = raw[TEXT_COL].astype(str).str.contains(r'science\s*direct', case=False, na=False).mean()
_ev = raw[TEXT_COL].astype(str).str.contains(r'\bevolve\b', case=False, na=False).mean()
print(f'\nProvenance: {_sd * 100:.1f}% of conversations name ScienceDirect, '
      f'{_ev * 100:.1f}% mention Evolve')
assert _sd > _ev, (
    'more conversations mention Evolve than ScienceDirect - this looks like the Evolve '
    'extract. See the note at the top of this notebook.')

# ---- 1. Filter to the target year --------------------------------------
# The parsed date becomes a column *before* filtering. Carrying it as a separate
# Series and using it after .reset_index() would realign on the old index and
# silently drop rows from any later .assign()/groupby.
raw['parsed_date'] = pd.to_datetime(raw[DATE_COL], errors='coerce')
assert raw['parsed_date'].isna().sum() == 0, f'unparseable dates in {DATE_COL}'

in_year = raw['parsed_date'].dt.year == TARGET_YEAR
excluded = raw[~in_year].copy()
excluded.to_csv(INTER_DIR / 'sd_ac_excluded_non2026.csv', index=False)
tickets = raw[in_year].copy().reset_index(drop=True)
assert len(tickets) + len(excluded) == len(raw), 'the year filter lost rows'

print(f'\nCalendar {TARGET_YEAR}: {len(tickets):,} kept, {len(excluded):,} excluded '
      f'({excluded["parsed_date"].min().date()} .. {excluded["parsed_date"].max().date()})')
print(f'  excluded rows written to {(INTER_DIR / "sd_ac_excluded_non2026.csv").relative_to(BASE)}')

_by_month = tickets['parsed_date'].dt.to_period('M').value_counts().sort_index()
print(f'\nKept, by month ({tickets["parsed_date"].min().date()} .. '
      f'{tickets["parsed_date"].max().date()}, '
      f'{tickets["parsed_date"].dt.date.nunique()} distinct days):')
print(_by_month.to_string())
assert _by_month.sum() == len(tickets), 'month counts do not sum to the kept rows'

print('\nBy channel:')
print(tickets['Channel'].value_counts().to_string())

In [ ]:
# ---- 2. Short-ticket rule ----------------------------------------------
_MARKER_RE = re.compile(r'(?mi)^\s*(customer|agent)(\s+notes)?\s*:')


def payload_len(text) -> int:
    """Conversation length ignoring the speaker markers themselves."""
    return len(_MARKER_RE.sub('', str(text)).strip())


tickets['conversation_chars_original'] = tickets[TEXT_COL].astype(str).str.len()
tickets['payload_chars'] = tickets[TEXT_COL].map(payload_len)
is_short = tickets['payload_chars'] < SHORT_CHAR_THRESHOLD

rule_assigned = tickets[is_short].copy()
rule_assigned['classification_source'] = 'rule:short_conversation'
rule_assigned.to_csv(INTER_DIR / 'sd_ac_rule_assigned.csv', index=False)
to_classify = tickets[~is_short].copy().reset_index(drop=True)

print(f'Short-ticket rule (payload < {SHORT_CHAR_THRESHOLD} chars): '
      f'{len(rule_assigned):,} rows set aside, {len(to_classify):,} to classify')
print('  most common content-free values:')
print(rule_assigned[TEXT_COL].astype(str).str.slice(0, 70).value_counts().head(8).to_string())

# ---- 3. Truncate long conversations ------------------------------------
TRUNC_MARKER = '\n\n[...TRUNCATED {n:,} CHARS FROM MIDDLE...]\n\n'


def truncate_middle(text) -> str:
    """Keep the head and a weighted tail, marking the cut visibly."""
    s = str(text)
    if len(s) <= TRUNC_CAP:
        return s
    dropped = len(s) - TRUNC_HEAD - TRUNC_TAIL
    return s[:TRUNC_HEAD] + TRUNC_MARKER.format(n=dropped) + s[-TRUNC_TAIL:]


to_classify['conversation_sent'] = to_classify[TEXT_COL].map(truncate_middle)
_n_trunc = int((to_classify['conversation_chars_original'] > TRUNC_CAP).sum())
_kept_share = (to_classify['conversation_sent'].str.len().sum()
               / max(to_classify['conversation_chars_original'].sum(), 1))
_max_sent = int(to_classify['conversation_sent'].str.len().max())
print(f'\nTruncation: {_n_trunc} of {len(to_classify):,} rows cut at {TRUNC_CAP:,} chars; '
      f'{_kept_share * 100:.1f}% of all text kept '
      f'(longest original {int(to_classify["conversation_chars_original"].max()):,} chars)')
assert _max_sent <= TRUNC_CAP + len(TRUNC_MARKER.format(n=0)) + 12, \
    f'a conversation of {_max_sent:,} chars survived the cap'

# ---- 4. Write the stage-1 input ----------------------------------------
# Two columns only. Metadata is re-joined in section 13 - carrying it through the
# pipeline risks it being stripped as an enrichment column.
stage1_df = to_classify[[ID_COL, 'conversation_sent']].rename(
    columns={'conversation_sent': TEXT_COL})
stage1_df.to_csv(STAGE1_INPUT, index=False)
assert len(stage1_df) == len(to_classify) and stage1_df[ID_COL].is_unique
print(f'\nWrote {STAGE1_INPUT.relative_to(BASE)}: {len(stage1_df):,} rows, '
      f'{STAGE1_INPUT.stat().st_size / 1e6:.0f} MB')
print(f'Population accounting: {len(rule_assigned):,} by rule + {len(stage1_df):,} to the LLM '
      f'= {len(rule_assigned) + len(stage1_df):,} (kept {len(tickets):,})')
assert len(rule_assigned) + len(stage1_df) == len(tickets), 'populations do not reconcile'


## 6. Generate the profiles and arm the fingerprint guard

`rca run` applies a profile key only when the matching CLI arg is still at its parser
default, via a `hasattr` guard — so a **key it does not recognise is silently ignored**. Writing
`"max_char"` for `"max_chars"` would not error, it would quietly use the 60,000 default. Every
emitted key is therefore validated against the real argparse destinations first, read off
`--help` rather than regex-scanned out of the source: that is the same list argparse will use, and
it doubles as a smoke test that `rca run` imports cleanly under this interpreter.

The tuning itself is set in the section 0 config cell — `batch_size 5`, `max_chars 40000`,
`concurrency 4`, `retries 5`, `max_tokens 8192`, `temperature 0.0` — rather than here, because
section 4 folds part of it into the checkpoint fingerprints and a constant read in section 4 cannot
be defined in section 6.

**`temperature` is pinned at 0.0 in the profile.** `RunConfig` already defaults to that via
`RCA_TEMPERATURE`, so this changes no behaviour — but a default is not a record. Written into the
profile, the value that produced the dataset sits next to the path it was written to, and a
future change to the default cannot silently alter a re-run.

It is worth being clear about what pinning it does **not** buy. Temperature 0.0 is not
determinism. Two runs of the 30-ticket smoke on an identical prompt gave 7 and then 8
`Accessing content`, and one ticket moved between `PURCHASED` and `OTHER`. Two causes, neither
fixed by temperature: records are sent five to a request, so a ticket's neighbours differ between
runs and it is read in a different context; and the backend is not bit-reproducible even at 0.0.
Treat single-run counts as carrying about ±1 ticket of noise at n=30.

**The fingerprint now covers the tuning, not just the prompt.** `temperature`, `max_tokens`,
`batch_size` and `max_chars` all change what comes back — the last two because they decide which
records share a request — so editing any of them invalidates the checkpoint exactly as a prompt
edit does. `concurrency`, `retries` and the cost-tracking flags are excluded, so tuning throughput
is free.

In [ ]:
# ---- Which keys does rca run actually accept? -------------------
_help = subprocess.run([sys.executable, '-m', 'rca_pipeline.cli', 'run', '--help'],
                       text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
assert _help.returncode == 0, f'rca run --help failed:\n{_help.stderr[-2000:]}'
_dests = {f.replace('-', '_') for f in re.findall(r'--([a-z][a-z0-9-]+)', _help.stdout)}
assert {'input', 'output', 'batch_size', 'max_chars', 'modules'} <= _dests, \
    f'could not parse rca run flags (got {sorted(_dests)})'



def write_profile(path: Path, *, comment, input_rel, output_rel, modules) -> Path:
    spec = {'_comment': comment, 'input': input_rel, 'output': output_rel,
            'taxonomy_dir': 'generated/taxonomies',
            'text_col': TEXT_COL, 'id_col': ID_COL, 'modules': modules, **TUNING}
    unknown = [k for k in spec if k != '_comment' and k not in _dests]
    assert not unknown, f'{path.name}: keys rca run would silently ignore: {unknown}'
    path.write_text(json.dumps(spec, indent=2) + '\n', encoding='utf-8')
    return path


write_profile(
    PROFILE_P,
    comment=(f'Stage 1 of 2 - primary classification ({ACCESSING_CONTENT} vs {ANYTHING_ELSE}) '
             f'for every {TARGET_YEAR} ScienceDirect ticket. Taxonomy transcribed from the '
             f'Accessing Content specification via {DEFN_XLSX.name}. Input is written by '
             f'notebooks/10_sd_2026_accessing_content.ipynb ({TARGET_YEAR}-filtered, short '
             f'tickets removed, conversations capped at {TRUNC_CAP} chars) - not the raw file.'),
    input_rel=STAGE1_INPUT.relative_to(BASE).as_posix(),
    output_rel=(OUT_DIR / 'sd_2026_ac_step1_primary.xlsx').relative_to(BASE).as_posix(),
    modules=[MOD_P])

write_profile(
    PROFILE_S,
    comment=(f'Stage 2 of 2 - sub-contact reason, run only on the subset stage 1 classified as '
             f'{ACCESSING_CONTENT}. The input path is overridden per call; the value here is the '
             f'full-run subset written by section 11.'),
    input_rel=(INTER_DIR / 'sd_ac_stage2_input.csv').relative_to(BASE).as_posix(),
    output_rel=(OUT_DIR / 'sd_2026_ac_step2_subreason.xlsx').relative_to(BASE).as_posix(),
    modules=[MOD_S])

for _p in (PROFILE_P, PROFILE_S):
    print(f'--- {_p.relative_to(BASE)}')
    print(_p.read_text(encoding='utf-8'))

In [ ]:
CHECKPOINT_DIR = BASE / 'runs' / 'checkpoints'


def checkpoint_files(input_path) -> list:
    """The checkpoint artefacts rca run keeps for a given input file."""
    stem = Path(input_path).stem
    return [CHECKPOINT_DIR / f'{stem}_checkpoint.jsonl',
            CHECKPOINT_DIR / f'{stem}_bad_uids.txt']


def _fingerprint_file(input_path) -> Path:
    return CHECKPOINT_DIR / f'{Path(input_path).stem}_checkpoint.fingerprint'


def _read_fingerprints(fp: Path) -> dict:
    """Read a {module: hash} fingerprint file, tolerating older formats.

    Earlier notebooks wrote a single bare 16-char hash here, which is not valid
    JSON. Treat anything unparseable, or a plain string, as 'no fingerprint' so the
    checkpoint is discarded rather than crashing the run.
    """
    if not fp.exists():
        return {}
    try:
        stored = json.loads(fp.read_text(encoding='utf-8'))
    except (json.JSONDecodeError, UnicodeDecodeError):
        return {}
    return stored if isinstance(stored, dict) else {}


def _sync_checkpoint(input_path, module: str) -> None:
    """Drop this input's checkpoint if it was produced by a different prompt.

    The pipeline resumes from runs/checkpoints/<stem>_checkpoint.jsonl whenever
    every module already has a result for a record - it logs `skip=N todo=0` and
    makes no LLM calls. That is what you want after an interruption and exactly
    what you do not want after editing the taxonomy.
    """
    want = PROMPT_FINGERPRINTS.get(module)
    if want is None:
        return
    fp = _fingerprint_file(input_path)
    stored = _read_fingerprints(fp)
    if stored.get(module) == want:
        return
    dropped = [p.name for p in checkpoint_files(input_path) if p.exists()]
    for p in checkpoint_files(input_path):
        p.unlink(missing_ok=True)
    if dropped:
        print(f'  {module}: prompt changed since that checkpoint '
              f'({stored.get(module) or "no fingerprint"} -> {want}); '
              f'discarded {", ".join(dropped)}')


def _stamp_checkpoint(input_path, module: str) -> None:
    want = PROMPT_FINGERPRINTS.get(module)
    if want is None:
        return
    fp = _fingerprint_file(input_path)
    fp.parent.mkdir(parents=True, exist_ok=True)
    stored = _read_fingerprints(fp)
    stored[module] = want
    fp.write_text(json.dumps(stored, indent=2), encoding='utf-8')


def run_stage(profile: Path, *, module: str, input_path=None, output_path=None,
              label: str = '', fresh: bool = False) -> Path:
    """Run one pipeline stage as a subprocess and return its output path.

    A subprocess rather than an in-process call so a dropped kernel does not kill a
    four-hour run, and so MODULE_REGISTRY is loaded fresh from the current config.

    fresh=True deletes the checkpoint first, forcing real LLM calls. Use it for the
    smoke test, whose whole purpose is to exercise the path: a smoke test that
    replays cache proves nothing. Never use it for the full run in sections 9-10,
    where resuming a four-hour job is the point.
    """
    spec = json.loads(profile.read_text(encoding='utf-8'))
    in_path = Path(input_path) if input_path else BASE / spec['input']
    out_path = Path(output_path) if output_path else BASE / spec['output']
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if fresh:
        gone = [p.name for p in checkpoint_files(in_path) if p.exists()]
        for p in checkpoint_files(in_path):
            p.unlink(missing_ok=True)
        _fingerprint_file(in_path).unlink(missing_ok=True)
        if gone:
            print(f'  fresh=True: discarded {", ".join(gone)} so this call really runs')
    else:
        _sync_checkpoint(in_path, module)
    cmd = [sys.executable, '-m', 'rca_pipeline.cli', 'run',
           '--profile', str(profile),
           '--input', str(in_path), '--output', str(out_path),
           '--modules', module]
    print(f'\n{"=" * 78}\n{label or module}\n{"=" * 78}')
    print(f'$ {" ".join(cmd)}\n')
    _t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding='utf-8', errors='replace',
                            bufsize=1, env=SUBPROCESS_ENV)
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'{label or module} failed (exit {rc})')
    assert out_path.exists(), f'{label or module} reported success but wrote no output'
    _stamp_checkpoint(in_path, module)
    print(f'\n{label or module}: {time.time() - _t0:.0f}s -> {out_path.relative_to(BASE)}')
    return out_path


print('Stage runner and checkpoint guard defined.')
print(f'Fingerprints armed: {PROMPT_FINGERPRINTS}')

### Stage-2 routing

The pipeline has no way to pass stage 1's answer into a stage 2 prompt, so the routing is done at
the data layer: take the ids stage 1 gave the `Accessing content` code, re-join their text, and run
the sub-reason module over that subset alone. The specification's rule — *do not assign a
sub-contact reason to tickets classified as Anything else* — then holds **by construction** rather
than by asking the model to respect it.

Routing is keyed on the **code**, never the label. A label comparison is one invisible character
away from an empty subset, and an empty subset here means every `Accessing content` ticket quietly
finishes with no sub-reason and no error.

`text_df` is always the truncated stage-1 input, so stage 2 sees exactly the text stage 1 saw.
Text is never read back out of a stage-1 `.xlsx`: openpyxl silently truncates cells at 32,767
characters, which would hand stage 2 a shortened transcript for the longest tickets — and 49 of
these conversations are longer than that.

In [ ]:
def run_stage2_over(pred_primary: pd.DataFrame, text_df: pd.DataFrame, *,
                    out_dir: Path, tag: str = '', label: str = 'stage 2',
                    fresh: bool = False) -> pd.DataFrame:
    """Run the sub-reason module over the tickets stage 1 called Accessing content.

    pred_primary needs [ID_COL, 'primary_code']; text_df needs [ID_COL, TEXT_COL].
    Returns one row per routed ticket; tickets stage 1 sent elsewhere are absent
    by design, and section 11 asserts that.
    """
    ids = pred_primary.loc[pred_primary['primary_code'] == AC_CODE, ID_COL].astype(str).str.strip()
    subset = text_df[text_df[ID_COL].astype(str).str.strip().isin(set(ids))]
    subset = subset[[ID_COL, TEXT_COL]].drop_duplicates(subset=[ID_COL])
    print(f'{label}: {len(subset):,} of {len(pred_primary):,} tickets routed to {MOD_S} '
          f'({len(subset) / max(len(pred_primary), 1) * 100:.1f}%)')
    assert len(subset) == len(set(ids)), (
        f'{len(set(ids)) - len(subset)} routed ticket(s) have no text - the re-join is broken')
    if subset.empty:
        return pd.DataFrame(columns=[ID_COL, 'sub_code', 'sub_reason', 'sub_spec_code',
                                     'sub_justification', 'sub_evidence'])

    in_path = out_dir / f'sd_ac_stage2_input{tag}.csv'
    # Written before the checkpoint is inspected: rca run keys the checkpoint
    # on record content, and this file is the population it will be compared against.
    subset.to_csv(in_path, index=False)
    _cp = checkpoint_files(in_path)[0]
    _before = sum(1 for _ in _cp.open(encoding='utf-8')) if _cp.exists() and not fresh else 0
    out_path = run_stage(PROFILE_S, module=MOD_S, input_path=in_path,
                         output_path=out_dir / f'sd_ac_step2{tag}.xlsx', label=label,
                         fresh=fresh)
    # A checkpoint carried over from an earlier run can cover every record in this
    # one, in which case no LLM call is made. That is correct - identical prompt,
    # identical text - but it must not be mistaken for a fresh measurement.
    if _before:
        _ids = set(subset[ID_COL].astype(str).str.strip())
        _cached = sum(1 for line in _cp.open(encoding='utf-8')
                      if json.loads(line).get('uid') in _ids)
        print(f'  {_cached} of {len(subset)} answers came from the checkpoint '
              f'({_before} records in it, from this and earlier runs)')
        if _cached == len(subset):
            print('  -> NO LLM CALLS were made. Same prompt and same text, so the answers '
                  'stand,\n     but this is a replay, not a re-run. Pass fresh=True to force '
                  'real calls.')

    got = read_stage_output(out_path)
    got['sub_code'] = got[f'{STUB_S}_code'].map(normalise_sub_code)
    got['sub_reason'] = got['sub_code'].map(lambda c: SUB_TAX_LABELS.get(c, ''))
    got['sub_spec_code'] = got['sub_code'].map(lambda c: SPEC_CODE_OF_TAX_CODE.get(c, ''))

    # What the model actually sent, before normalisation, so a recovered answer
    # is visible rather than looking like it arrived clean.
    _raw = got[f'{STUB_S}_code'].astype(str).str.strip()
    _recovered = _raw[(_raw != '') & (_raw != got['sub_code']) & (got['sub_code'] != '')]
    if len(_recovered):
        print(f'  {len(_recovered)} answer(s) mapped onto a leaf code: '
              f'{dict(_recovered.value_counts())}')
    _unknown = _raw[(_raw.str.lower().isin(('', 'nan')) == False) & (got['sub_code'] == '')]
    if len(_unknown):
        print(f'  {len(_unknown)} answer(s) NOT recognised: {dict(_unknown.value_counts())}')

    keep = {f'{STUB_S}_justification': 'sub_justification',
            f'{STUB_S}_evidence': 'sub_evidence'}
    # Azure refuses some transcripts outright (ContentPolicyViolationError). The
    # reason has to travel or those rows arrive in the deliverable as blanks with
    # no explanation.
    if '_error' in got.columns:
        keep['_error'] = 'sub_error'
    # rca run checks every quoted evidence string against the transcript
    # and adds <stub>_evidence_grounded. That is the closest thing to an automatic
    # check on a verbatim-quote field, so it travels into the deliverable.
    if f'{STUB_S}_evidence_grounded' in got.columns:
        keep[f'{STUB_S}_evidence_grounded'] = 'sub_evidence_grounded'
    out = got.rename(columns=keep)[[ID_COL, 'sub_code', 'sub_reason', 'sub_spec_code',
                                    *keep.values()]]

    # Reindex onto the subset that was sent. A resumed run omits records on the
    # bad-uid skip list, so 6,476 routed tickets came back as 6,466 rows - which
    # would make the routing assertion in section 10 fire on what is really a
    # disclosure. Return one row per ticket given, and say why the blanks are blank.
    _missing = [u for u in subset[ID_COL] if u not in set(out[ID_COL])]
    if _missing:
        _bad_file = checkpoint_files(in_path)[1]
        _known_bad = set()
        if _bad_file.exists():
            _known_bad = {u.strip() for u in _bad_file.read_text(encoding='utf-8').split()
                          if u.strip()}
        _pad = pd.DataFrame({ID_COL: _missing})
        for _c in ('sub_code', 'sub_reason', 'sub_spec_code',
                   'sub_justification', 'sub_evidence'):
            if _c in out.columns:
                _pad[_c] = ''
        _pad['sub_error'] = [
            'skipped: on the bad-uid list, the backend refuses this text'
            if u in _known_bad else 'absent from the stage output, cause unrecorded'
            for u in _missing]
        out = pd.concat([out, _pad], ignore_index=True)
        print(f'  restored {len(_missing)} record(s) the pipeline had skipped '
              f'({len([u for u in _missing if u in _known_bad])} on the bad-uid list)')

    assert len(out) == len(subset), f'{len(out)} rows returned for {len(subset)} sent'
    return out


def _leaf_lookup(leaf_labels_map: dict, spec_of: dict | None = None) -> dict:
    """Every code form the model might answer with, mapped to the real leaf code.

    Three forms are accepted, all unambiguous for these taxonomies:
      - the leaf code itself, e.g. SDAS6.1
      - the specification code carried in the label, e.g. OTHER -> SDAS6.1.
        Putting the spec code in the label makes the answer legible, but it also
        invites the model to answer with it; 2 of 6,476 did.
      - the L1 group code, e.g. SDAS6 -> SDAS6.1. Every group here has exactly
        one child, so this cannot be ambiguous. The prompt already tells the
        model to use leaf codes; this is the safety net.

    Anything else maps to '' and is reported, never guessed at.
    """
    out = {}
    for leaf in leaf_labels_map:
        out[leaf.upper()] = leaf
        out[leaf.split('.')[0].upper()] = leaf
        if spec_of and leaf in spec_of:
            out[str(spec_of[leaf]).upper()] = leaf
    return out


def normalise_sub_code(value) -> str:
    """Stage-2 code as returned -> a valid leaf code, or '' if unrecognisable."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    key = str(value).strip().upper()
    if key in ('', 'NAN', 'NONE'):
        return ''
    return _SUB_CODE_FORMS.get(key, '')


def normalise_primary_code(value) -> str:
    """Same for stage 1. There are no spec codes here, only leaf and L1 forms."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    key = str(value).strip().upper()
    if key in ('', 'NAN', 'NONE'):
        return ''
    return _PRIMARY_CODE_FORMS.get(key, '')


_SUB_CODE_FORMS = _leaf_lookup(SUB_TAX_LABELS, SPEC_CODE_OF_TAX_CODE)
_PRIMARY_CODE_FORMS = _leaf_lookup(PRIMARY_LABELS)


def read_primary(path) -> pd.DataFrame:
    """Read a stage-1 output and attach the decoded primary label.

    Renames the pipeline's generic `_error` to `<stub>_error` so it survives the
    merge in section 12 alongside stage 2's own, rather than colliding with it.
    """
    df = read_stage_output(path)
    df['primary_code'] = df[f'{STUB_P}_code'].map(normalise_primary_code)
    df['primary_class'] = df['primary_code'].map(lambda c: PRIMARY_LABELS.get(c, ''))
    if '_error' in df.columns:
        df = df.rename(columns={'_error': f'{STUB_P}_error'})
    return df


print('Stage-2 router defined.')
print(f'Stage-2 code forms accepted: {sorted(_SUB_CODE_FORMS)}')

## 7. Gate 3 — 30-ticket smoke test *(~$0.25)*

Drawn from the production population and stratified over `Channel`, because a phone note and a web
form read nothing like each other and 30 web forms would prove only that web forms parse. Three
sampled from each of the six largest channels, then a random fill to 30, all under a fixed seed so
the sample is reproducible.

**Sampled, not sliced.** The input is date-ordered, and templated request types — Patient Access
form submissions especially — arrive in batches, so taking the first three rows of a channel
returns three near-identical tickets from the same hour. An earlier version of this cell did
exactly that and drew 19 of its 30 tickets from the first 600 rows of 22,041, which made Patient
Access 5 of 8 routed tickets against a population keyword rate of 19.2%. The cell now asserts the
sample spans at least four months.

The point is not accuracy at n=30 — it is to prove the plumbing works end to end: valid JSON, no
blank codes, codes that exist in the taxonomy, routing into stage 2 that actually fires, and
justifications that read like reasoning rather than a restatement of the label.

Both stages here run with `fresh=True`, which deletes the checkpoint first. A smoke test that
replays cached answers proves nothing about the path it is meant to be testing, so every
re-run of this section costs its ~$0.22 again. That is deliberate. The full run in sections 9
and 10 does the opposite and resumes, because there a dropped four-hour job is the risk.

**This is the gate that matters, and it is a human reading rather than a number.** There is no
labelled set to score against (see *What this notebook does not tell you* at the top), so the 30
justifications and evidence quotes below are the evidence on which ~$165 gets committed. Read them.

In [ ]:
SMOKE_N = 30
SMOKE_PER_CHANNEL = 3
SMOKE_SEED = 0

# .sample(), not .head(). The file is date-ordered, so head(n) returns adjacent
# tickets from the same hour - and templated request types such as Patient Access
# arrive in batches, so a contiguous slice over-represents whichever type happened
# to be arriving. An earlier version of this cell used head(3) and drew 19 of its
# 30 tickets from the first 600 rows of 22,041, with 7 from rows 0-6.
_strata = to_classify['Channel'].fillna('(none)')
_top = _strata.value_counts().head(6).index.tolist()
_parts = [to_classify[_strata == s].sample(
              n=min(SMOKE_PER_CHANNEL, int((_strata == s).sum())),
              random_state=SMOKE_SEED)
          for s in _top]
_seeded = pd.concat(_parts)
_rest = (to_classify[~to_classify[ID_COL].isin(set(_seeded[ID_COL]))]
         .sample(n=SMOKE_N - len(_seeded), random_state=SMOKE_SEED))
smoke_tickets = pd.concat([_seeded, _rest]).reset_index(drop=True)
assert smoke_tickets[ID_COL].is_unique and len(smoke_tickets) == SMOKE_N

smoke_in = SMOKE_DIR / 'sd_ac_smoke_input.csv'
smoke_tickets[[ID_COL, 'conversation_sent']].rename(
    columns={'conversation_sent': TEXT_COL}).to_csv(smoke_in, index=False)
print(f'{len(smoke_tickets)} tickets across {smoke_tickets["Channel"].nunique()} channels:')
print(smoke_tickets['Channel'].value_counts().to_string())

# Is the sample actually spread? A clustered sample is the failure this cell had
# before, and it is invisible unless printed.
_pos = {r: i for i, r in enumerate(stage1_df[ID_COL])}
_rows = sorted(_pos[r] for r in smoke_tickets[ID_COL] if r in _pos)
_months = smoke_tickets['parsed_date'].dt.to_period('M').value_counts().sort_index()
print(f'\nSpread check - row positions within the {len(stage1_df):,}-row input:')
print(f'  min {_rows[0]:,}  median {_rows[len(_rows) // 2]:,}  max {_rows[-1]:,}')
print(f'  in the first 1,000 rows: {sum(1 for r in _rows if r < 1000)} of {len(_rows)}')
print(f'  distinct months covered: {len(_months)} of '
      f'{tickets["parsed_date"].dt.to_period("M").nunique()}')
print('  ' + _months.to_string().replace('\n', '\n  '))
assert len(_months) >= 4, (
    'the smoke sample spans fewer than 4 months - it is clustered, and a gate on a '
    'clustered sample tells you about one week rather than the population')

# fresh=True: a smoke test that replays a checkpoint proves nothing about the
# path it is supposed to be testing.
smoke_out = run_stage(PROFILE_P, module=MOD_P, input_path=smoke_in,
                      output_path=SMOKE_DIR / 'smoke_step1.xlsx',
                      label=f'SMOKE stage 1 - {len(smoke_tickets)} tickets', fresh=True)
smoke_p = read_primary(smoke_out)

_blank = int((smoke_p['primary_class'] == '').sum())
_bad = sorted(set(smoke_p.loc[smoke_p['primary_class'] == '', 'primary_code']) - {''})
print(f'\nBlank/invalid primary codes: {_blank}' + (f'  unknown codes: {_bad}' if _bad else ''))
assert _blank == 0, 'stage 1 produced codes outside the taxonomy - fix before spending more'
print(smoke_p['primary_class'].value_counts().to_string())

print('\n--- by channel ---')
print(pd.crosstab(smoke_tickets.set_index(ID_COL)['Channel']
                  .reindex(smoke_p[ID_COL]).values,
                  smoke_p['primary_class'].values).to_string())

In [ ]:
# Read a few justifications and evidence quotes: reasoning, or restating the label?
for _, r in smoke_p.head(8).iterrows():
    _ev = r[f'{STUB_P}_evidence']
    _ev = '' if pd.isna(_ev) else str(_ev)[:220]
    print(f'--- {r[ID_COL]}  {r["primary_code"]}  {r["primary_class"]}')
    print(f'    why     : {r[f"{STUB_P}_justification"]}')
    print(f'    evidence: {_ev or "(none offered)"}')

In [ ]:
smoke_s = run_stage2_over(smoke_p[[ID_COL, 'primary_code']],
                          smoke_tickets[[ID_COL, 'conversation_sent']].rename(
                              columns={'conversation_sent': TEXT_COL}),
                          out_dir=SMOKE_DIR, tag='_smoke', label='SMOKE stage 2',
                          fresh=True)


def show(value, width=220) -> str:
    """Printable form of a field the model may have left empty.

    An empty evidence string comes back through pandas as NaN and would print as
    the bare word 'nan', which reads like the model said something.
    """
    return '' if pd.isna(value) else str(value)[:width]


if not smoke_s.empty:
    print('\nSub-contact reasons assigned:')
    print(smoke_s['sub_spec_code'].replace('', '(blank)').value_counts().to_string())
    assert int((smoke_s['sub_reason'] == '').sum()) == 0, 'blank sub-reason codes in the smoke run'
    _no_quote = int(smoke_s['sub_evidence'].map(lambda v: show(v).strip() == '').sum())
    print(f'Rows with no evidence quote: {_no_quote} of {len(smoke_s)}')
    for _, r in smoke_s.head(8).iterrows():
        print(f'\n--- {r[ID_COL]}  {r["sub_spec_code"]}  {r["sub_reason"]}')
        print(f'    why     : {show(r["sub_justification"])}')
        print(f'    evidence: {show(r["sub_evidence"]) or "(none offered)"}')
else:
    print('No ticket in the smoke sample was classified Accessing content - '
          'enlarge the sample before reading anything into stage 2.')

### Decision point

Everything past here costs about $165, and there is no accuracy figure to justify it — see *What
this notebook does not tell you* at the top of the notebook. The judgement is a reading, not a
threshold. Continue when:

- the 30 justifications above **reason from the transcript** rather than restating the label, and
  the evidence quotes are things the customer or the agent actually wrote;
- the ones you disagree with are cases where the **specification** is ambiguous, not cases where
  the prompt has missed a rule it was given. If it is the latter, edit section 1, re-run sections
  1–4 and 7, and come back — that costs $0.25, not $165;
- the split between the two classes is not degenerate. Everything landing in one class means the
  prompt is not discriminating, whatever the individual justifications look like.

Set `STOP_AFTER_SMOKE = False` once you have read the output above.

In [ ]:
STOP_AFTER_SMOKE = False   # set False after reading the smoke output above

print(f'Primary classification across {len(smoke_p)} smoke tickets:')
print(smoke_p['primary_class'].value_counts().to_string())
if not smoke_s.empty:
    print(f'\nSub-contact reasons across the {len(smoke_s)} routed tickets:')
    print(smoke_s['sub_spec_code'].value_counts().to_string())

if STOP_AFTER_SMOKE:
    raise SystemExit('STOP_AFTER_SMOKE is True - read the justifications and evidence above, '
                     'then set it False and re-run from here.')

## 8. Gate 4 — 1,000-record scale probe *(~$6)*

30 read-by-hand tickets to 22,041 is a 735x jump onto text that has been truncated and
rule-filtered. This measures the two things a 30-ticket read cannot: **throughput** on real
batches, and the **blank-code rate** — the share of tickets that come back with no usable code.

`--limit` is not a `rca run` flag, so the slice is written as its own input file. It gets
its own checkpoint, which is why it does not contaminate the full run.

It also prints the predicted `Accessing content` share — the first honest estimate of how much
stage 2 will cost — and how that share splits by channel. Channel matters here: 41% of this
population is `CSS Email` and 40% is `Web`, and a web form states its subject far more plainly than
a phone note written up after the call. A large split between the two is not an error, but it is
worth understanding before it is baked into 23,000 rows.

One caveat on the throughput figure: **it is only a measurement the first time this cell runs.**
Re-run it and the checkpoint resumes, no LLM call is made, and the elapsed time becomes disk-read
speed — the cell prints ~99 records/second and a projection of 0.0 hours. The cell detects that and
says so rather than letting the number stand. To time it again for real, delete
`runs/checkpoints/sd_ac_probe_input_*` first.

In [ ]:
PROBE_N = 1000
probe_in = SMOKE_DIR / 'sd_ac_probe_input.csv'
_stage1_all = pd.read_csv(STAGE1_INPUT, dtype=str)
_stage1_all.sample(n=PROBE_N, random_state=0).to_csv(probe_in, index=False)

_t0 = time.time()
probe_out = run_stage(PROFILE_P, module=MOD_P, input_path=probe_in,
                      output_path=SMOKE_DIR / 'probe_step1.xlsx',
                      label=f'PROBE stage 1 - {PROBE_N} production tickets')
_elapsed = time.time() - _t0

probe = read_primary(probe_out)
_blank_rate = float((probe['primary_class'] == '').mean())
_rate = len(probe) / _elapsed

# A resumed run reads its answers from the checkpoint and makes no LLM calls, so
# _rate would then be measuring disk. Anything above ~10 rec/s on this backend is
# a cache read, not throughput, and the projection built on it is meaningless.
_RESUMED = _rate > 10
_n_full = len(_stage1_all)
print(f'\nThroughput   : {_rate:.2f} records/sec')
print(f'Blank codes  : {int((probe["primary_class"] == "").sum())} / {len(probe)} '
      f'({_blank_rate * 100:.2f}%)')
if _RESUMED:
    print('\n*** This cell resumed from its checkpoint and made no LLM calls, so the ***')
    print('*** throughput above is cache-read speed, not a measurement. Delete      ***')
    print('*** runs/checkpoints/sd_ac_probe_input_* to time it for real.            ***')
    print('Measured on the first run: 1.31 rec/s -> 4.7 h for stage 1, $0.00556/ticket.')
else:
    print(f'Projected stage 1 on {_n_full:,} records: {_n_full / _rate / 3600:.1f} h')

_share = probe['primary_class'].replace('', '(blank)').value_counts(normalize=True) * 100
print('\nPrimary classification on production text:')
print(_share.round(1).to_string())
_ac_share = float((probe['primary_class'] == ACCESSING_CONTENT).mean())
print(f'\n{ACCESSING_CONTENT}: {_ac_share * 100:.1f}%  '
      f'-> stage 2 will run on roughly {int(_ac_share * _n_full):,} tickets'
      + ('' if _RESUMED else f', ~{_ac_share * _n_full / _rate / 3600:.1f} h at this rate'))

assert _blank_rate <= 0.005, (
    f'blank-code rate {_blank_rate * 100:.2f}% exceeds 0.5% - investigate before the full run')
print('\nBlank-code rate within tolerance.')

In [ ]:
# The probe rows carry only an id and text, so join the channel back on.
# A description of where the classification lands, not a score.
_probe_channel = tickets.set_index(ID_COL)['Channel']
_pc = pd.crosstab(probe[ID_COL].map(_probe_channel).fillna('(unknown)'),
                  probe['primary_class'])
_pc['Accessing content %'] = (_pc.get(ACCESSING_CONTENT, 0)
                              / _pc.sum(axis=1) * 100).round(1)
print('Predicted primary classification by channel (probe, n=1,000):')
print(_pc.sort_values('Accessing content %', ascending=False).to_string())

### Is the quoted evidence real?

`rca run` checks every `*_evidence` string against the transcript it came from and sets
`*_evidence_grounded` accordingly — it flags a quote it cannot find rather than blanking it. On a
field whose whole purpose is to be **verbatim**, that is the one automatic check available, and it
is the closest thing this notebook has to a guard on the specification's "do not infer" rule.

Two things about the flag are worth knowing before reading the number, both found by chasing a
`robust_json_loads: escaped-smart-quote retry succeeded` warning in a probe log.

**Evidence is verbatim except for quote characters.** `robust_json_loads` rewrites `“ ” ‘ ’` to
ASCII on *every* response before parsing it, so a quote the model copied faithfully arrives with
straight quotes where the transcript has typographic ones. In one 1,000-record probe, 397
transcripts contained a curly quote and **zero** evidence strings did. No prompt change can affect
this — the rewrite happens after the model has answered.

**The check used to be biased against non-Latin scripts.** Its fallback tier measured overlap over
whitespace tokens, which is close to meaningless in Chinese and Japanese: a 50-character quote is
one token, so one differing character read as a fabrication. Measured on the same probe, CJK
evidence was flagged at 9.9% against 0.32% for everything else. `rca_pipeline/utils.py` now folds
typographic punctuation before matching and falls back to character overlap for low-token text,
which brought CJK to 1.4% while leaving other scripts untouched at 0.32%. The cell below therefore
reports the rate **by script**, because one aggregate hid that entirely.

So read the flagged rows rather than the rate. Some are real paraphrase; some are a quote drawn
from the middle of a conversation that was truncated before the model saw it.

In [ ]:
_GROUND = f'{STUB_P}_evidence_grounded'
_CJK_RE = re.compile(r'[\u3000-\u9fff\uff00-\uffef]')   # CJK, kana, full-width forms

if _GROUND not in probe.columns:
    print(f'{_GROUND} not present - the grounding check did not run on this stage.')
else:
    _g = probe[_GROUND]
    _ev_col = probe[f'{STUB_P}_evidence'].fillna('').astype(str)
    _n_true = int((_g == True).sum())           # noqa: E712 - nullable boolean column
    _n_false = int((_g == False).sum())         # noqa: E712
    _n_na = len(probe) - _n_true - _n_false
    print(f'Evidence verified present in the transcript : {_n_true} / {len(probe)} '
          f'({_n_true / len(probe) * 100:.1f}%)')
    print(f'Quoted but not found                       : {_n_false}')
    print(f'No quote offered                           : {_n_na}')

    # Split by script. A single rate hid a 31x disparity: the check's fallback
    # tier measured overlap over whitespace tokens, and Chinese and Japanese
    # barely use whitespace, so a 50-character quote counted as one token and a
    # single differing character read as a fabrication. Fixed in
    # rca_pipeline.utils on 2026-09-01; figures from runs before that date
    # overstate the flag rate for non-Latin scripts.
    _cjk = _ev_col.map(lambda t: bool(_CJK_RE.search(t)))
    print(f'\n{"script":<10} {"n":>6} {"flagged":>8} {"rate":>8}')
    for _name, _mask in (('CJK', _cjk), ('other', ~_cjk)):
        _sub = probe[_mask]
        if not len(_sub):
            continue
        _f = int((_sub[_GROUND] == False).sum())        # noqa: E712
        print(f'{_name:<10} {len(_sub):>6} {_f:>8} {_f / len(_sub) * 100:>7.2f}%')
    print('A gap of more than a couple of points between these two is a property of')
    print('the check, not of the model - read the flagged rows before believing it.')

    if _n_false:
        print('\nFlagged rows, with the class they were quoted for:')
        for _, _r in probe[_g == False].head(10).iterrows():   # noqa: E712
            print(f'  {_r[ID_COL]}  {_r["primary_class"]:<18} '
                  f'{str(_r[f"{STUB_P}_evidence"])[:90]}')

## 9. Stage 1 — primary classification for every 2026 ticket *(~$125, ~4.7 h)*

Runs as a subprocess, so a dropped kernel does not kill it. If it is interrupted, **re-run this
cell**: the checkpoint resumes and only unfinished records are paid for again. The fingerprint
guard means that resume is safe *unless* the prompt changed, in which case the checkpoint is
discarded rather than silently reused.

In [ ]:
step1_out = run_stage(PROFILE_P, module=MOD_P, label='FULL stage 1 - primary classification')
step1 = read_primary(step1_out)

stage1_input = pd.read_csv(STAGE1_INPUT, dtype=str)
stage1_input[ID_COL] = stage1_input[ID_COL].astype(str).str.strip()

# ---- Restore records the pipeline skipped -------------------------------
# rca run records permanently-failing uids in <stem>_bad_uids.txt and
# skips them on subsequent runs. So the first run emits them as stubs with no
# code, and a resumed run omits them altogether: 22,041 rows became 22,031.
# Reindex onto the input so the population does not depend on which of those two
# happened, and say why each missing row is missing.
_bad_uids_file = checkpoint_files(STAGE1_INPUT)[1]
_known_bad = set()
if _bad_uids_file.exists():
    _known_bad = {u.strip() for u in _bad_uids_file.read_text(encoding='utf-8').split()
                  if u.strip()}

_missing = [u for u in stage1_input[ID_COL] if u not in set(step1[ID_COL])]
if _missing:
    _restored = pd.DataFrame({ID_COL: _missing})
    _restored['primary_code'] = ''
    _restored['primary_class'] = ''
    _restored[f'{STUB_P}_justification'] = ''
    _restored[f'{STUB_P}_evidence'] = ''
    _restored[f'{STUB_P}_error'] = [
        'skipped: on the bad-uid list, the backend refuses this text'
        if u in _known_bad else 'absent from the stage output, cause unrecorded'
        for u in _missing]
    step1 = pd.concat([step1, _restored], ignore_index=True)
    print(f'Restored {len(_missing)} record(s) the pipeline had skipped '
          f'({len([u for u in _missing if u in _known_bad])} of them on the bad-uid list).')

assert len(step1) == len(stage1_input), (
    f'row count still wrong: {len(step1)} out, {len(stage1_input)} in')
assert step1[ID_COL].is_unique, 'duplicate ticket ids in the stage-1 output'
assert set(step1[ID_COL]) == set(stage1_input[ID_COL]), 'stage 1 changed the population'

print(f'\nRows: {len(step1):,}')
print('\nPrimary classification:')
for label, n in step1['primary_class'].replace('', '(no answer)').value_counts().items():
    print(f'  {n:>6}  {n / len(step1) * 100:>5.1f}%  {label}')

_blank = step1[step1['primary_class'] == '']
print(f'\nNo answer: {len(_blank)} ({len(_blank) / len(step1) * 100:.2f}%)')
if f'{STUB_P}_error' in step1.columns and len(_blank):
    _kinds = _blank[f'{STUB_P}_error'].fillna('(no reason recorded)').astype(str).str.slice(0, 70)
    for _kind, _n in _kinds.value_counts().items():
        print(f'  {_n:>4}  {_kind}')
    print('  A content-filter refusal returns the same result on a retry, so these are '
          'disclosed\n  in the deliverable rather than re-run.')
assert len(_blank) / len(step1) <= 0.005, 'blank rate above 0.5% - investigate'


## 10. Stage 2 — sub-contact reason for the Accessing content subset *(~$40, ~1.4 h)*

`run_stage2_over` selects on the **primary code**, re-joins the truncated text from
`data/sd_ac_stage1_input_2026.csv`, and runs the sub-reason module over that subset alone.

Three assertions close the loop on the specification's own rules:

- every routed ticket got a valid leaf code of `sd_ac_subreason`;
- exactly the tickets stage 1 called `Accessing content` have a sub-reason — no more, no fewer;
- no ticket classified `Anything else` carries one.

The third is the specification's explicit prohibition. It holds by construction here, but it is
asserted anyway: a future change to the routing helper would otherwise break it silently.

In [ ]:
sub_reasons = run_stage2_over(step1[[ID_COL, 'primary_code']], stage1_input,
                              out_dir=INTER_DIR, tag='', label='FULL stage 2')

_ac_ids = set(step1.loc[step1['primary_code'] == AC_CODE, ID_COL])
_got_ids = set(sub_reasons[ID_COL])
assert _got_ids == _ac_ids, (
    f'routing mismatch: {len(_ac_ids - _got_ids)} Accessing content ticket(s) got no sub-reason, '
    f'{len(_got_ids - _ac_ids)} non-Accessing-content ticket(s) got one')

_sub_blank = int((sub_reasons['sub_reason'] == '').sum())
print(f'\nSub-reasons assigned: {len(sub_reasons) - _sub_blank:,} of {len(sub_reasons):,}')

# Two different things, previously conflated by one assertion.
#
# An unrecognised code is a BUG: the model answered something the taxonomy
# cannot express and normalise_sub_code could not map. That must be zero.
_assigned = sub_reasons.loc[sub_reasons['sub_code'] != '', 'sub_code']
_invalid = sorted(set(_assigned) - set(SUB_TAX_LABELS))
assert not _invalid, (
    f'stage 2 returned code(s) outside the taxonomy that could not be mapped: {_invalid}. '
    f'Extend _leaf_lookup if this is a new legitimate answer form.')

# A missing code is a DELIVERY FACT: the model returned nothing for that record.
# In this run every one was an Azure content-filter refusal, which a retry cannot
# fix, so they are disclosed rather than treated as a failure to be re-run.
if 'sub_error' in sub_reasons.columns:
    _errs = sub_reasons.loc[sub_reasons['sub_error'].notna(), 'sub_error'].astype(str)
    print(f'Records the model returned nothing for: {_sub_blank} '
          f'({_sub_blank / max(len(sub_reasons), 1) * 100:.2f}%), of which '
          f'{len(_errs)} carry an error from the backend:')
    for _kind, _n in _errs.str.slice(0, 60).value_counts().items():
        print(f'  {_n:>4}  {_kind}')
else:
    print(f'Records the model returned nothing for: {_sub_blank} '
          f'({_sub_blank / max(len(sub_reasons), 1) * 100:.2f}%)')
assert _sub_blank / max(len(sub_reasons), 1) <= 0.005, (
    f'{_sub_blank} of {len(sub_reasons)} routed tickets have no sub-reason - above the 0.5% '
    f'tolerance, so this is worth investigating rather than disclosing')

print('\nDistribution:')
_vc = sub_reasons['sub_spec_code'].replace('', '(blank)').value_counts()
for _code, _n in _vc.items():
    print(f'  {_code:<14} {_n:>6}  {_n / len(sub_reasons) * 100:>5.1f}% of Accessing content')

## 11. Marker audit — the only quantitative check stage 2 has *(free)*

The specification does not merely define the five specific sub-contact reasons, it states the
**markers that must be present** for each. That turns into something testable: for every
prediction, does the transcript actually contain a marker belonging to the code the model chose?

The patterns below are taken from the specification's own marker lists, widened for spelling
variation (`licence`/`license`, `digitised`/`digitized`), for the inflections English forces
(`subscrib`, `entitle`), and for the ways customers actually identify an institution. That last
one is not a liberty: the specification introduces each list with *"Markers include:"*, which is
explicitly non-exhaustive.

**It was a liberty in the other direction that had to be corrected.** The first version of these
lists required the possessive `our institution` / `our library`, exactly as the document writes
them, and had no pattern for an institution simply being named. So a ticket opening *"The
Townsville University Hospital, where I work, has full text access"* counted as having no
institutional marker at all. That reported `INST_ISSUE` at 35.9% unsupported and `INST_INFO` at
23.9% — both apparently over-firing. With the institution identifiers below they are **16.4% and
6.0%**, inside the guidance, and the model was right about them all along.

Worth stating plainly, since it has now happened three times in this notebook: when this audit
disagrees with the model, suspect the audit first. The definitions were **not** changed — they are
verbatim specification paragraphs and section 1 asserts it.

They deliberately do **not** try to reproduce the model's judgement — a match proves a word is
present, not that the classification is right.

Three things this catches:

- **Over-firing.** A specific code chosen with none of its markers in the text is the model
  inferring, which the specification forbids in as many words.
- **`PA` in particular.** The specification singles it out: Patient Access must be *explicit*, and
  must not be assumed for healthcare professionals. `PA` is audited twice — once for its own
  markers, and once for whether the word *patient* appears at all.
- **The reverse case, for `OTHER`.** A ticket the model called `OTHER` whose transcript is full of
  invoice and entitlement language is a *missed* specific code, and gets counted separately.

This is the only quantitative signal stage 2 gets, because no labelled set exists for the six
sub-contact reasons.

Interpretation, so this is not over-read: an unsupported rate under ~10% on a code is noise —
markers get paraphrased, and the notebook is matching the truncated text. A rate above ~25% on one
specific code means that code is over-firing; fix its definition in section 1 and re-run stage 2
only. The audit is a **flag column in the deliverable, never a correction**: nothing here silently
overwrites a model answer.

In [ ]:
# Marker patterns from the specification's "Definition and required markers"
# column, keyed by the specification's own code.
#
# The specification introduces each list with "Markers include:", which is
# explicitly non-exhaustive, so these are widened for the ways customers
# actually write. That is not licence to invent: everything below is a
# restatement of a marker the document names, in a form real tickets use.

# How a customer identifies an institution. The specification names
# "our institution" and "our library"; in practice "my institution" is just as
# common, and most tickets simply name the place. Requiring the possessive
# "our" made the audit report 36% of INST_ISSUE as unsupported when the
# institution was named in the first line.
_INST_IDENTIFIERS = [
    r'\b(our|my|the)\s+(institution|library|univers\w*|hospital|college|school)',
    r'\bunivers\w+\b', r'\bhospital\b', r'\bcollege\b', r'\buniversidad\b',
    r'where\s+I\s+(work|study)', r'\bI\s+work\s+(at|for)\b',
    r'\binstitutional\s+access\b',
    # Access routes that only exist because an institution provides them.
    r'\bcampus\b', r'\bproxy\b', r'\bathens\b', r'\bshibboleth\b',
]

SUB_MARKERS = {
    'PA': [
        r'patient\s*access', r'\bpatient\b', r'\bcaregiver\b', r'\bcarer\b',
        r'non-?professional',
    ],
    'INST_INFO': [
        r'\bsubscri', r'\blicen[cs]e', r'\bentitle', r'\bholdings?\b', r'\bcoverage\b',
        r'\bKBART\b', r'title\s+inclusion', r'\bpackage\b', r'\bconsorti',
        *_INST_IDENTIFIERS,
    ],
    'INST_ISSUE': [
        r'we\s+are\s+subscri', r'we\s+have\s+paid', r'we\s+should\s+have\s+access',
        r'we\s+(subscribe|have\s+access|are\s+entitled)',
        r'\binvoice', r'\bagreement\b', r'\bentitle', r'\bECR\b', r'account\s+mapping',
        r'institutional\s+token', r'provisioned', r'\bsubscri', r'\bwe\s+pay\b',
        *_INST_IDENTIFIERS,
    ],
    'PURCHASED': [
        r'\border\s*(number|no\.?|#|id)', r'\breceipt\b', r'\bpayment\b', r'\bpaid\b',
        r'pay[\s-]?per[\s-]?view', r'\bPPV\b', r'\bpurchas', r'\bbought\b',
        r'order\s+history', r'personal\s+subscription', r'\bmy\s+subscription\b',
        r'\bcredit\s*card\b', r'\binvoice\b',
    ],
    'AVAILABILITY': [
        r'\bunavailable\b', r'\bnot\s+available\b', r'\bmissing\b', r'\bnot\s+digiti[sz]ed\b',
        r'\bsupplement', r'\bcompanion\s+file', r'\bappendi', r'\bdataset\b',
        r'\bsupporting\s+(material|information|file)', r'\bnot\s+found\b',
        r'\bno\s+longer\s+(available|published)', r'\barchive\b', r'\bcannot\s+be\s+supplied\b',
    ],
}
# OTHER has no required markers by definition - it is the code for their absence.
SUB_MARKERS['OTHER'] = []

_COMPILED = {k: [re.compile(p, re.I) for p in v] for k, v in SUB_MARKERS.items()}
_SPECIFIC = [k for k in SUB_MARKERS if k != 'OTHER']

print('Marker patterns audited (from the specification, widened only for spelling and')
print('inflection):\n')
for _k, _v in SUB_MARKERS.items():
    print(f'  {_k:<14} {len(_v):>2} pattern(s)')
    for _p in _v:
        print(f'                 {_p}')
    print()

In [ ]:
def markers_hit(text: str, spec_code: str) -> list:
    """Which of this code's markers appear in the text."""
    return [p.pattern for p in _COMPILED.get(spec_code, []) if p.search(text)]


# Audit against the same (truncated) text the model saw, not the full conversation:
# a marker present only in the discarded middle was never available to the model.
_text_seen = stage1_input.set_index(ID_COL)[TEXT_COL].astype(str)

audit = sub_reasons[[ID_COL, 'sub_spec_code', 'sub_reason', 'sub_evidence']].copy()
audit['text'] = audit[ID_COL].map(_text_seen).fillna('')
audit['markers_found'] = [markers_hit(t, c)
                          for t, c in zip(audit['text'], audit['sub_spec_code'])]
audit['n_markers'] = audit['markers_found'].map(len)
audit['unsupported'] = (audit['n_markers'] == 0) & audit['sub_spec_code'].isin(_SPECIFIC)

# The specification's extra requirement for PA: Patient Access must be explicit.
_patient = re.compile(r'\bpatient', re.I)
audit['pa_without_patient'] = ((audit['sub_spec_code'] == 'PA')
                               & ~audit['text'].map(lambda t: bool(_patient.search(t))))

# The reverse: OTHER on a ticket whose text carries some specific code's markers.
def _other_candidates(text: str) -> list:
    return [c for c in _SPECIFIC if markers_hit(text, c)]


_is_other = audit['sub_spec_code'] == 'OTHER'
audit['other_with_markers_for'] = ''
audit.loc[_is_other, 'other_with_markers_for'] = (
    audit.loc[_is_other, 'text'].map(lambda t: ','.join(_other_candidates(t))))

# ---- Report -------------------------------------------------------------
MARKER_AUDIT = (audit.groupby('sub_spec_code')
                .agg(tickets=(ID_COL, 'size'),
                     unsupported=('unsupported', 'sum'),
                     mean_markers=('n_markers', 'mean'))
                .reindex([c for c, _n, _d in SUB_DEFS]).dropna(subset=['tickets']))
MARKER_AUDIT['unsupported_%'] = (MARKER_AUDIT['unsupported']
                                 / MARKER_AUDIT['tickets'] * 100).round(1)
MARKER_AUDIT['mean_markers'] = MARKER_AUDIT['mean_markers'].round(2)
MARKER_AUDIT = MARKER_AUDIT.astype({'tickets': int, 'unsupported': int})
print('Unsupported predictions - a specific code chosen with none of its own markers')
print('present in the text the model was shown:\n')
print(MARKER_AUDIT.to_string())

_pa_bad = int(audit['pa_without_patient'].sum())
_pa_n = int((audit['sub_spec_code'] == 'PA').sum())
print(f'\nPA without the word "patient" anywhere in the text: {_pa_bad} of {_pa_n}')
print('  The specification requires Patient Access to be explicit, so this count should be 0.')

_other_hits = audit.loc[_is_other & (audit['other_with_markers_for'] != ''),
                        'other_with_markers_for']
print(f'\nOTHER on text that does carry a specific code\'s markers: '
      f'{len(_other_hits)} of {int(_is_other.sum())}')
if len(_other_hits):
    print(_other_hits.value_counts().head(10).to_string())
    print('  These are possible missed specific codes, not confirmed errors - the markers may')
    print('  appear in agent boilerplate rather than in the customer\'s request.')

for _code in _SPECIFIC:
    _bad = audit[(audit['sub_spec_code'] == _code) & audit['unsupported']]
    if _bad.empty:
        continue
    print(f'\n--- {_code}: {len(_bad)} unsupported, first 3 ---')
    for _, r in _bad.head(3).iterrows():
        print(f'  {r[ID_COL]}  quoted evidence: {str(r["sub_evidence"])[:160]}')

## 12. Merge everything back onto the 23,055 tickets

Two populations reunite here, and `classification_source` keeps them distinguishable:

| source | how the labels were assigned |
|---|---|
| `llm` | primary classification by model; sub-reason by model where routed |
| `rule:short_conversation` | `Anything else` by rule — the conversation carried no content |

That column exists because "how many tickets had nothing usable in them?" is a question ops will
ask, and without it the rule-assigned rows are indistinguishable from a model decision.

The untruncated conversation is re-attached here, so the deliverable carries the real text even
where the model saw a shortened version; `was_truncated` says which rows those are.

In [ ]:
# -- 1. LLM-classified tickets --------------------------------------------
_s1_cols = [ID_COL, 'primary_code', 'primary_class',
            f'{STUB_P}_justification', f'{STUB_P}_evidence']
for _opt in (f'{STUB_P}_evidence_grounded', f'{STUB_P}_error'):
    if _opt in step1.columns:
        _s1_cols.append(_opt)
merged = step1[_s1_cols].rename(columns={
    f'{STUB_P}_justification': 'primary_justification',
    f'{STUB_P}_evidence': 'primary_evidence',
    f'{STUB_P}_evidence_grounded': 'primary_evidence_grounded'})
merged = merged.merge(sub_reasons, on=ID_COL, how='left')

# Most rows are plain 'llm'. Records the backend refused outright get their own
# source so they are separable in the deliverable: a blank primary class with no
# explanation is indistinguishable from a bug, and a reader is entitled to know
# the difference. Azure's content filter refuses the same text on a retry, so
# these will not resolve by re-running.
merged['classification_source'] = 'llm'
_p_err = (merged[f'{STUB_P}_error'].notna()
          if f'{STUB_P}_error' in merged.columns else pd.Series(False, index=merged.index))
_p_blank = merged['primary_class'].fillna('') == ''
merged.loc[_p_blank & ~_p_err, 'classification_source'] = 'error:no_primary_class'
merged.loc[_p_err, 'classification_source'] = 'error:refused_stage1'
if 'sub_error' in merged.columns:
    merged.loc[merged['sub_error'].notna(), 'classification_source'] = 'error:refused_stage2'

# Attach the marker-audit flags. NA for Anything else rows: not "passed the audit",
# but "was never audited", which is a different statement.
merged = merged.merge(
    audit[[ID_COL, 'n_markers', 'unsupported', 'pa_without_patient',
           'other_with_markers_for']].rename(columns={
               'n_markers': 'markers_found_n',
               'unsupported': 'marker_audit_unsupported',
               'pa_without_patient': 'marker_audit_pa_not_explicit',
               'other_with_markers_for': 'marker_audit_other_has_markers_for'}),
    on=ID_COL, how='left')

# -- 2. Rule-assigned short tickets ---------------------------------------
rule_rows = pd.read_csv(INTER_DIR / 'sd_ac_rule_assigned.csv', dtype=str)
rule_rows[ID_COL] = rule_rows[ID_COL].astype(str).str.strip()
rule_frame = pd.DataFrame({
    ID_COL: rule_rows[ID_COL],
    'primary_code': CODE_OF_PRIMARY[ANYTHING_ELSE],
    'primary_class': ANYTHING_ELSE,
    'primary_justification': (
        f'Conversation payload is under {SHORT_CHAR_THRESHOLD} characters and states nothing that '
        f'could evidence a content request, so it was assigned by rule without an LLM call. The '
        f'specification offers no third outcome and forbids inference.'),
    'primary_evidence': '',
    'primary_evidence_grounded': pd.NA,
    'sub_code': '', 'sub_reason': '', 'sub_spec_code': '',
    'sub_justification': '', 'sub_evidence': '',
    'markers_found_n': pd.NA, 'marker_audit_unsupported': pd.NA,
    'marker_audit_pa_not_explicit': pd.NA, 'marker_audit_other_has_markers_for': '',
    'classification_source': 'rule:short_conversation',
})

final = pd.concat([merged, rule_frame], ignore_index=True)
for col in ('sub_code', 'sub_reason', 'sub_spec_code', 'sub_justification', 'sub_evidence',
            'marker_audit_other_has_markers_for'):
    final[col] = final[col].fillna('')
# The grounding flags stay nullable on purpose: NA means no quote was offered,
# which is a different statement from False (quoted, but not found in the text).

# -- 3. Re-attach metadata and the untruncated conversation ---------------
meta = tickets[[ID_COL, 'Incident_ID', 'Channel',
                DATE_COL, 'parsed_date', 'conversation_chars_original', TEXT_COL]].copy()
meta[ID_COL] = meta[ID_COL].astype(str).str.strip()
final = meta.merge(final, on=ID_COL, how='left')
final['month'] = final['parsed_date'].dt.to_period('M').astype(str)
final['was_truncated'] = final['conversation_chars_original'] > TRUNC_CAP

# -- 4. Reconcile ---------------------------------------------------------
assert len(final) == len(tickets), f'{len(final)} rows out, {len(tickets)} tickets in'
assert final[ID_COL].is_unique, 'duplicate ids after the merge'
assert int(final['primary_class'].isna().sum()) == 0, 'a ticket ended with no primary class'
_no_sub = final[(final['primary_class'] == ANYTHING_ELSE) & (final['sub_reason'] != '')]
assert _no_sub.empty, (
    f'{len(_no_sub)} ticket(s) classified {ANYTHING_ELSE} carry a sub-contact reason - '
    f'the specification forbids this')

print(f'Final rows: {len(final):,}')
print('\nBy classification source:')
print(final['classification_source'].value_counts().to_string())
print('\nPrimary classification (all 2026 tickets):')
for _label, _n in final['primary_class'].value_counts().items():
    print(f'  {_n:>6}  {_n / len(final) * 100:>5.1f}%  {_label}')
print(f'\nSub-contact reason (of the {int((final["sub_reason"] != "").sum()):,} '
      f'{ACCESSING_CONTENT} tickets):')
_sub_only = final[final['sub_reason'] != '']
for _code, _n in _sub_only['sub_spec_code'].value_counts().items():
    print(f'  {_code:<14} {_n:>6}  {_n / len(_sub_only) * 100:>5.1f}%')

## 13. Write the deliverable

Ten sheets. `fit_excel` truncates over-long cells **visibly** rather than letting openpyxl do it
silently at 32,767 characters, and the companion CSV carries the untruncated text.

`Taxonomy` holds the specification's definitions once, rather than repeating them on 23,055 rows,
and marks which two rules were authored rather than transcribed.

`Eval methodology` is not optional. Anyone reading only the workbook must be able to see that
**no accuracy figure exists for either stage**, that every number in it is a count or a
distribution, and that the marker audit is a flag rather than a verdict.

In [ ]:
XL_LIMIT = 32_767
MARKER = '  ...[TRUNCATED FOR EXCEL - see companion CSV for full text]'

OUT_COLS = [ID_COL, 'Incident_ID', 'Channel', DATE_COL, 'month',
            'primary_code', 'primary_class', 'primary_justification', 'primary_evidence',
            'sub_code', 'sub_spec_code', 'sub_reason', 'sub_justification', 'sub_evidence',
            'markers_found_n', 'marker_audit_unsupported', 'marker_audit_pa_not_explicit',
            'marker_audit_other_has_markers_for',
            'classification_source',
            'conversation_chars_original', 'was_truncated', TEXT_COL]
for _col, _after in (('primary_evidence_grounded', 'primary_evidence'),
                     ('sub_evidence_grounded', 'sub_evidence')):
    if _col in final.columns and _col not in OUT_COLS:
        OUT_COLS.insert(OUT_COLS.index(_after) + 1, _col)

deliverable = final[OUT_COLS].copy()
deliverable.to_csv(FINAL_CSV, index=False, encoding='utf-8-sig')
print(f'Wrote {FINAL_CSV.relative_to(BASE)} ({len(deliverable):,} rows, untruncated text)')


def fit_excel(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    long = df[TEXT_COL].astype(str).str.len() > XL_LIMIT
    df.loc[long, TEXT_COL] = (df.loc[long, TEXT_COL].astype(str)
                              .str[:XL_LIMIT - len(MARKER)] + MARKER)
    return df


def summary(df: pd.DataFrame, col: str, denom: int) -> pd.DataFrame:
    vc = df[col].replace('', '(none)').value_counts()
    return pd.DataFrame({col: vc.index, 'tickets': vc.values,
                         'share_%': (vc.values / max(denom, 1) * 100).round(1)})


# -- Primary summary, with a complexity proxy -----------------------------
primary_sum = summary(deliverable, 'primary_class', len(deliverable))
primary_sum['mean_conversation_chars'] = primary_sum['primary_class'].map(
    deliverable.groupby('primary_class')['conversation_chars_original'].mean().round(0))
primary_sum['rule_assigned'] = primary_sum['primary_class'].map(
    deliverable[deliverable['classification_source'] != 'llm']
    ['primary_class'].value_counts()).fillna(0).astype(int)

# -- Sub-reason summary: share of Accessing content AND of everything -----
_ac = deliverable[deliverable['sub_reason'] != '']
sub_sum = summary(_ac, 'sub_reason', len(_ac)).rename(
    columns={'share_%': 'share_of_accessing_content_%'})
sub_sum.insert(0, 'spec_code', sub_sum['sub_reason'].map(
    lambda lab: SUB_CODE_OF_LABEL.get(lab, '')))
sub_sum['share_of_all_tickets_%'] = (sub_sum['tickets'] / len(deliverable) * 100).round(2)
sub_sum['mean_conversation_chars'] = sub_sum['sub_reason'].map(
    _ac.groupby('sub_reason')['conversation_chars_original'].mean().round(0))
sub_sum['marker_unsupported_%'] = sub_sum['spec_code'].map(
    MARKER_AUDIT['unsupported_%'] if len(MARKER_AUDIT) else {})

sub_by_month = pd.crosstab(_ac['sub_spec_code'], _ac['month'])
sub_by_channel = pd.crosstab(_ac['sub_spec_code'], _ac['Channel'])
primary_by_channel = pd.crosstab(deliverable['Channel'].fillna('(unknown)'),
                                 deliverable['primary_class'])
primary_by_channel['accessing_content_%'] = (
    primary_by_channel.get(ACCESSING_CONTENT, 0)
    / primary_by_channel.sum(axis=1) * 100).round(1)
primary_by_month = pd.crosstab(deliverable['month'], deliverable['primary_class'])

# -- Taxonomy sheet -------------------------------------------------------
_tax_rows = [{'stage': '1 primary', 'code': CODE_OF_PRIMARY[lab], 'spec_code': '',
              'label': lab, 'definition': PRIMARY_DEFS[lab], 'source': 'specification, verbatim'}
             for lab in PRIMARY_ORDER]
_tax_rows += [{'stage': '2 sub-reason', 'code': CODE_OF_SUB[lab],
               'spec_code': SUB_CODE_OF_LABEL[lab], 'label': lab, 'definition': d,
               'source': ('specification, verbatim' if c != 'OTHER' else
                          'specification; sentence completed from the Classification principles '
                          'block because the table cell is cut off mid-sentence')}
              for (c, _n, d), lab in zip(SUB_DEFS, SUB_LABELS)]
_tax_rows += [
    {'stage': '1 primary', 'code': '', 'spec_code': '', 'label': 'AUTHORED - decision order',
     'definition': PRIMARY_PRECEDENCE,
     'source': 'AUTHORED for this run. Not in the specification. See notebook section 1.'},
    {'stage': '2 sub-reason', 'code': '', 'spec_code': '',
     'label': 'AUTHORED - evidence threshold', 'definition': SUB_EVIDENCE_RULE,
     'source': 'AUTHORED for this run. Not in the specification. See notebook section 1.'},
    {'stage': 'both', 'code': '', 'spec_code': '', 'label': 'Classification principles',
     'definition': PRINCIPLES, 'source': 'specification, verbatim'},
]
taxonomy_sheet = pd.DataFrame(_tax_rows)

# -- Marker audit sheet ---------------------------------------------------
marker_sheet = MARKER_AUDIT.reset_index().rename(columns={'sub_spec_code': 'spec_code'})
marker_sheet['patterns_checked'] = marker_sheet['spec_code'].map(
    lambda c: ' | '.join(SUB_MARKERS.get(c, [])) or '(none - OTHER is the absence of markers)')

# -- Eval methodology sheet ----------------------------------------------
_eval_rows = [
    ('Population', f'{len(deliverable):,} ScienceDirect tickets dated {TARGET_YEAR} '
                   f'({deliverable[DATE_COL].min()} to {deliverable[DATE_COL].max()}) from '
                   f'{TICKETS_CSV.name}. The file also holds {len(excluded):,} rows dated '
                   f'2023-2025, which were excluded from this run.'),
    ('Accuracy - NONE MEASURED',
     'No accuracy, precision, recall or F1 figure is quoted anywhere, because no human-labelled '
     'set exists for this taxonomy. The specification is new and nothing carries Accessing '
     'content / Anything else labels, let alone the six sub-contact reasons. Every number in this '
     'workbook is a COUNT or a DISTRIBUTION. None is a measure of correctness and none should be '
     'read as one.'),
    ('What was checked instead',
     'The exact prompts were reviewed against the specification before the run, with every '
     'definition asserted present verbatim. A 30-ticket sample was read by hand for whether the '
     'justifications reason from the transcript rather than restating the label. A 1,000-ticket '
     'probe measures the blank-code rate and throughput before the full run - see the notebook '
     'section 8 output for the figures that applied to THIS run. Evidence grounding and the '
     'marker audit are described below.'),
    ('How to get an accuracy figure',
     'Hand-label a few hundred tickets sampled from the Tickets sheet, stratified on the predicted '
     'sub_spec_code so the rare codes (PA, AVAILABILITY) are represented rather than swamped, and '
     'score against them. That is the only thing that turns this from a described dataset into a '
     'measured one.'),
    ('Stage 2 evidence',
     'The Marker audit sheet and a hand read of the 30 smoke-test tickets are the only evidence '
     'of stage-2 quality. Both are described in this sheet; neither is a score.'),
    ('Evidence grounding',
     (f'{int((final.get(chr(112) + "rimary_evidence_grounded", pd.Series(dtype=object)) == True).sum()):,} '
      f'of {len(final):,} primary-classification quotes were found verbatim in the transcript by '
      f'rca run. It flags a quote it cannot locate rather than blanking it. Some flagged '
      f'quotes are real but were drawn from the middle of a conversation that was truncated before '
      f'the model saw it, or differ only by CJK normalisation - read them rather than counting '
      f'them.')),
    ('Evidence is verbatim except for quote characters',
     'The JSON parser rewrites typographic quotes to ASCII on every model response before the '
     'result is read, so an evidence string carries straight quotes where the transcript may '
     'have curly ones. Nothing else about the quote is altered. Match on normalised punctuation '
     'if you compare evidence against source text.'),
    ('Evidence grounding is a measurement, not a verdict',
     'The grounded flag is a fuzzy substring check, and it is more forgiving of some scripts '
     'than others. Before a fix on 2026-09-01 it flagged CJK evidence at 9.9% against 0.32% for '
     'other scripts, almost entirely because its fallback counted whitespace tokens. It now '
     'reads 1.4% against 0.32%. A flagged row is worth reading, not deleting: a quote taken from '
     'the middle of a truncated conversation flags too.'),
    ('Marker audit meaning',
     'A flag, not a correction. It reports whether the transcript contains any of the markers '
     'the specification requires for the code the model chose. A match does not prove the '
     'classification is right, and a miss does not prove it wrong - markers get paraphrased. '
     'It is a rate to watch per code, and nothing in the dataset was changed because of it.'),
    ('Rule-assigned rows',
     f'{int((deliverable["classification_source"] != "llm").sum()):,} tickets have a payload '
     f'under {SHORT_CHAR_THRESHOLD} characters and were set to {ANYTHING_ELSE} by rule with no '
     f'LLM call. The specification has no third outcome and forbids inference, so a ticket '
     f'stating nothing cannot evidence a content request. Filter on classification_source to '
     f'exclude them.'),
    ('Text truncation',
     f'{int(deliverable["was_truncated"].sum())} conversation(s) exceeded {TRUNC_CAP:,} chars and '
     f'were sent to the model as head {TRUNC_HEAD:,} + tail {TRUNC_TAIL:,} with a visible marker. '
     f'The Conversation column in the CSV holds the full text. Separately, '
     f'{int((deliverable["conversation_chars_original"] >= 32_767).sum())} conversation(s) were '
     f'already cut at 32,767 chars in the source export and are incomplete at source.'),
    ('Authored rules',
     'Two prompt rules are not in the specification: a decision order for the primary '
     'classification, and an evidence threshold for the sub-reason. Both are printed in full on '
     'the Taxonomy sheet, marked AUTHORED.'),
    ('Input provenance',
     'The original input for this analysis was data/ScienceDirect_Collapsed.xlsx. That file is '
     'row-for-row identical to data/Evolve_Collapsed.xlsx (126,504 rows compared, 9 differ, '
     'and only by an empty cell written once as nan and once as an empty string); 79.5% of '
     'its 2026 conversations mention Evolve and 0.1% mention ScienceDirect. It was not used. '
     'This run uses data/sd_collapsed_2026.csv, in which 57.1% of conversations name '
     'ScienceDirect and 0.1% mention Evolve.'),
    ('Specification defects',
     ' | '.join(f'{d["where"]}: {d["issue"]} Resolved by: {d["resolution"]}'
                for d in SPEC_DEFECTS)),
]
eval_sheet = pd.DataFrame(_eval_rows, columns=['topic', 'detail'])

# -- Write ----------------------------------------------------------------
with pd.ExcelWriter(FINAL_XLSX, engine='openpyxl') as xw:
    fit_excel(deliverable).to_excel(xw, sheet_name='Tickets', index=False)
    primary_sum.to_excel(xw, sheet_name='Primary summary', index=False)
    sub_sum.to_excel(xw, sheet_name='Sub-reason summary', index=False)
    sub_by_month.to_excel(xw, sheet_name='Sub-reason x month')
    sub_by_channel.to_excel(xw, sheet_name='Sub-reason x channel')
    primary_by_channel.to_excel(xw, sheet_name='Primary x channel')
    primary_by_month.to_excel(xw, sheet_name='Primary x month')
    marker_sheet.to_excel(xw, sheet_name='Marker audit', index=False)
    taxonomy_sheet.to_excel(xw, sheet_name='Taxonomy', index=False)
    eval_sheet.to_excel(xw, sheet_name='Eval methodology', index=False)

print(f'Wrote {FINAL_XLSX.relative_to(BASE)}')
print(f'  Tickets            {len(deliverable):>6,} rows x {len(OUT_COLS)} cols')
print(f'  Primary summary    {len(primary_sum):>6} rows')
print(f'  Sub-reason summary {len(sub_sum):>6} rows')
print(f'  Marker audit       {len(marker_sheet):>6} rows')
print(f'  Taxonomy           {len(taxonomy_sheet):>6} rows')
print(f'  Eval methodology   {len(eval_sheet):>6} rows')
print()
print(primary_sum.to_string(index=False))
print()
print(sub_sum.to_string(index=False))

## Afterwards

### Restoring the ClinicalKey / CSD modules

`generated/pipeline_config.json` now describes the two ScienceDirect accessing-content modules, so
the `csd_*` ones are no longer loadable. Their outputs are untouched under `runs/output/csd_2026_*`.
To put them back:

```
rca build --input definitions/csd_2026_pipeline_definition.xlsx --force-all
```

That drops these two in turn — the config holds one product at a time. The backup taken in section
0c is in `generated/_backup_pre_sd_ac/`, alongside a `RESTORE.txt`.

### Re-running after a taxonomy edit

Edit section 1, then re-run sections 1, 2, 3, 4 and 6. The fingerprint check in section 6 discards
only the checkpoints of modules whose prompt actually changed, so a stage-2-only fix costs one
stage's worth of re-classification rather than the whole run.

An edit to `PRIMARY_PRECEDENCE` invalidates stage 1 and therefore stage 2 as well, because the
subset routed to stage 2 changes. An edit to `SUB_EVIDENCE_RULE` or to a sub-reason definition
invalidates stage 2 only — that is the cheap direction, roughly $40 rather than $170.

### Extending the run to 2025

The file holds 106,293 tickets dated 2025, already written out to
`runs/intermediate/sd_ac_excluded_non2026.csv`. Setting `TARGET_YEAR = 2025` in section 0 and
re-running sections 5 onward is all that is required; nothing in the taxonomy or the prompts is
year-specific. Budget roughly 5x this run.

### What would make stage 2 measurable

The single highest-value follow-up is **a labelled set for the six sub-contact reasons**. Around
150 `Accessing content` tickets labelled by hand — ideally stratified on the model's own predicted
code, so the rare ones (`PA`, `AVAILABILITY`) are represented — would turn section 12's marker
audit from a proxy into a real precision-and-recall measurement, and would let the definitions be
tuned against evidence instead of intuition. `runs/output/sd_2026_accessing_content.xlsx` is
already in the right shape to sample from.